In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyref.fitting as fit
import seaborn as sns
import uncertainties.unumpy as unp
from matplotlib import gridspec, ticker
from matplotlib.patches import Patch
from scienceplots import scienceplots
from uncertainties import ufloat, umath

from utils import read_fit, read_ooc, read_xrr
from utils.helpers.plotting_helper import set_plotting_defaults

plt.style.use(["science", "no-latex"])
set_plotting_defaults()


In [ ]:
dft_constrained = read_fit("dft/dft_en_offset_new2.pkl", material="znpc")
exp_constrained = read_fit("free/free_en_offset_init_2.pkl", material="znpc")


In [ ]:
ooc = read_ooc("dft.csv", material="znpc")
# Zoom from 270 to 300 eV
ooc = ooc[ooc["energy"] > 270]
ooc = ooc[ooc["energy"] < 300]

def name_map(s: str):
    match s:
        case "znpc":
            return "bulk"
        case "contamination":
            return "interface"
        case _:
            return s

def as_ufloat(p):
    if p.stderr is None:
        return ufloat(p.value, 0)
    return ufloat(p.value, p.stderr)

def rotate_tensor(xx, zz, ixx, izz, rotation):
    """
    n^r_xx = (n_xx * (1+c^2) + n_zz * s^2) / 2
    n^r_zz = n_xx * s^2 + n_zz * c^2
    """
    cs = umath.cos(rotation)**2
    ss = 1 - cs
    n_xx_r = (xx * (1+cs) + zz * ss) / 2
    n_zz_r = xx * ss + zz * cs
    n_ixx_r = (ixx * (1+cs) + izz * ss) / 2
    n_izz_r = ixx * ss + izz * cs
    return n_xx_r, n_zz_r, n_ixx_r, n_izz_r

def n(ooc_df, density, rotation, confidence=1):
    rotation = ufloat(rotation.n, rotation.s * confidence)
    density = ufloat(density.n, density.s * confidence)
    n_xx = unp.uarray(
        ooc_df["n_xx"].values,
        ooc_df["n_xx_err"].values if "n_xx_err" in ooc_df else np.zeros_like(ooc_df["n_xx"].values),
    )
    n_zz = unp.uarray(
        ooc_df["n_zz"].values,
        ooc_df["n_zz_err"].values if "n_zz_err" in ooc_df else np.zeros_like(ooc_df["n_zz"].values),
    )
    n_ixx = unp.uarray(
        ooc_df["n_ixx"].values,
        ooc_df["n_ixx_err"].values if "n_ixx_err" in ooc_df else np.zeros_like(ooc_df["n_ixx"].values),
    )
    n_izz = unp.uarray(
        ooc_df["n_izz"].values,
        ooc_df["n_izz_err"].values if "n_izz_err" in ooc_df else np.zeros_like(ooc_df["n_izz"].values),
    )

    n_o, n_e, n_io, n_ie = rotate_tensor(n_xx, n_zz, n_ixx, n_izz, rotation)
    n_o, n_e, n_io, n_ie = n_o * density, n_e * density, n_io * density, n_ie * density
    delta = 2 * n_o + n_e
    beta = 2 * n_io + n_ie
    dichroism = n_io - n_ie
    birefringence = n_o - n_e
    df = pd.DataFrame(
        {
            "energy": ooc_df["energy"].values,
            "n_o": unp.nominal_values(n_o),
            "n_o_err": unp.std_devs(n_o),
            "n_e": unp.nominal_values(n_e),
            "n_e_err": unp.std_devs(n_e),
            "n_io": unp.nominal_values(n_io),
            "n_io_err": unp.std_devs(n_io),
            "n_ie": unp.nominal_values(n_ie),
            "n_ie_err": unp.std_devs(n_ie),
            "delta": unp.nominal_values(delta),
            "delta_err": unp.std_devs(delta),
            "beta": unp.nominal_values(beta),
            "beta_err": unp.std_devs(beta),
            "dichroism": unp.nominal_values(dichroism),
            "dichroism_err": unp.std_devs(dichroism),
            "birefringence": unp.nominal_values(birefringence),
            "birefringence_err": unp.std_devs(birefringence),
        }
    )
    return df

def components(delta, biref):
    """
    Convert from (delta, birefringence) back to n_xx, n_zz.

    Here:
        delta        = 2*n_xx + n_zz
        biref        = n_xx - n_zz
    """
    n_xx = (delta + biref) / 3.0
    n_zz = (delta - 2.0 * biref) / 3.0
    return n_xx, n_zz

def extract_free_optical_parameters(objectives):
    params: dict[str, list] = {
        "slab": [],
        "energy": [],
        "delta": [],
        "delta_err": [],
        "beta": [],
        "beta_err": [],
        "dichroism": [],
        "dichroism_err": [],
        "birefringence": [],
        "birefringence_err": [],
        "xx": [],
        "xx_err": [],
        "zz": [],
        "zz_err": [],
        "ixx": [],
        "ixx_err": [],
        "izz": [],
        "izz_err": [],
    }

    for o in objectives:
        if o.model.energy == 250.0:
            continue
        structure = getattr(o.model.structure, "components", o.model.structure)
        for s in structure:
            if not hasattr(s, "sld") or isinstance(s.sld, fit.MaterialSLD):
                continue
            slab = name_map(s.name.split("_")[0].lower())
            xx = as_ufloat(s.sld.xx)
            zz = as_ufloat(s.sld.zz)
            ixx = as_ufloat(s.sld.ixx)
            izz = as_ufloat(s.sld.izz)
            delta = 2 * xx + zz
            beta = 2 * ixx + izz
            dich = ixx - izz
            biref = xx - zz
            params["slab"].append(slab)
            params["energy"].append(o.model.energy)
            params["delta"].append(delta.n)
            params["delta_err"].append(delta.s)
            params["beta"].append(beta.n)
            params["beta_err"].append(beta.s)
            params["dichroism"].append(dich.n)
            params["dichroism_err"].append(dich.s)
            params["birefringence"].append(biref.n)
            params["birefringence_err"].append(biref.s)
            params["xx"].append(xx.n)
            params["xx_err"].append(xx.s)
            params["zz"].append(zz.n)
            params["zz_err"].append(zz.s)
            params["ixx"].append(ixx.n)
            params["ixx_err"].append(ixx.s)
            params["izz"].append(izz.n)
            params["izz_err"].append(izz.s)

    return pd.DataFrame(params)

def extract_dft_optical_traces(dft_constrained, probe_energy, ooc, confidence=1):
    o = dft_constrained.objectives[
        np.where(np.array([o.model.energy for o in dft_constrained.objectives]) == probe_energy)[0][0]
    ]
    params = {
        "slab": [],
        "density": [],
        "rotation": [],
        "rotation_err": [],
        "energy": [],
        "xx": [],
        "xx_err": [],
        "zz": [],
        "zz_err": [],
        "ixx": [],
        "ixx_err": [],
        "izz": [],
        "izz_err": [],
        "delta": [],
        "delta_err": [],
        "beta": [],
        "beta_err": [],
        "dichroism": [],
        "dichroism_err": [],
        "birefringence": [],
        "birefringence_err": [],
    }
    structure = getattr(o.model.structure, "components", o.model.structure)
    for s in (structure if hasattr(structure, "__iter__") and not isinstance(structure, str) else [structure]):
        if isinstance(s.sld, fit.UniTensorSLD):
            slab = name_map(s.name.split("_")[0].lower())
            density = as_ufloat(s.sld.density)
            rotation = as_ufloat(s.sld.rotation)
            index = n(
                ooc,
                density,
                rotation,
                confidence=confidence,
            )
            params["slab"].extend([slab] * len(index))
            params["density"].extend([density] * len(index))
            params["rotation"].extend([rotation] * len(index))
            params["rotation_err"].extend([rotation.s] * len(index))
            params["energy"].extend(ooc["energy"].copy())
            params["xx"].extend(index["n_o"])
            params["xx_err"].extend(index["n_o_err"])
            params["zz"].extend(index["n_e"])
            params["zz_err"].extend(index["n_e_err"])
            params["ixx"].extend(index["n_io"])
            params["ixx_err"].extend(index["n_io_err"])
            params["izz"].extend(index["n_ie"])
            params["izz_err"].extend(index["n_ie_err"])
            params["delta"].extend(index["delta"])
            params["delta_err"].extend(index["delta_err"])
            params["beta"].extend(index["beta"])
            params["beta_err"].extend(index["beta_err"])
            params["dichroism"].extend(index["dichroism"])
            params["dichroism_err"].extend(index["dichroism_err"])
            params["birefringence"].extend(index["birefringence"])
            params["birefringence_err"].extend(index["birefringence_err"])
    df = pd.DataFrame(params)
    # Sort by slab to be ordered "surface", "bulk", "interface"
    df = pd.concat(
        [df[df["slab"] == "surface"], df[df["slab"] == "bulk"], df[df["slab"] == "interface"]])
    return df


In [ ]:
free_n = extract_free_optical_parameters(exp_constrained.objectives)
dft_n = extract_dft_optical_traces(dft_constrained, 283.7, ooc, confidence=1)

In [ ]:
def plot_optical_components(df, components=("delta", "beta", "dichroism", "birefringence")):
    slabs = list(df["slab"].unique())
    n_slabs = len(slabs)
    if n_slabs == 0:
        return

    fig_width = 3.35
    fig_height = 1.5 * n_slabs
    fig, axs = plt.subplots(
        n_slabs,
        1,
        figsize=(fig_width, fig_height),
        sharex=True,
        dpi=300,
        gridspec_kw={"hspace": 0.05},
    )

    if n_slabs == 1:
        axs = [axs]

    palette = sns.color_palette()
    for i, slab in enumerate(slabs):
        data = df[df["slab"] == slab].sort_values("energy")
        ax = axs[i]

        for j, comp in enumerate(components):
            color = palette[j % len(palette)]
            y = data[comp].to_numpy()
            yerr = data[f"{comp}_err"].to_numpy()

            ax.errorbar(
                data["energy"],
                y,
                yerr=yerr,
                label=comp,
                color=color,
                marker="o",
                linestyle="None",
                markersize=1.5,
                ecolor=color,
                capsize=0.5,
                elinewidth=0.5,
                markeredgecolor="k",
                markerfacecolor=color,
                markeredgewidth=0.3,
            )

        ax.set_ylabel(slab)
        ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

    axs[-1].set_xlabel("Energy (eV)")
    fig.align_labels()
    plt.show()

def plot_n_traces(
    dft_df,
    free_df,
    slabs=("surface", "bulk", "interface"),
    components=("xx", "zz")
)-> list[plt.Axes] | None:
    n_slabs = len(slabs)
    if n_slabs == 0:
        return

    fig_width = 3.35
    fig_height = 1.5 * n_slabs
    fig, axs = plt.subplots(
        n_slabs,
        1,
        figsize=(fig_width, fig_height),
        sharex=True,
        dpi=300,
        gridspec_kw={"hspace": 0.05},
    )

    if n_slabs == 1:
        axs = [axs]

    palette = sns.color_palette()
    for i, slab in enumerate(slabs):
        dft_slab = dft_df[dft_df["slab"] == slab].sort_values("energy")
        free_slab = free_df[free_df["slab"] == slab].sort_values("energy")
        ax = axs[i]

        for j, comp in enumerate(components):
            color = palette[j % len(palette)]
            err_col = f"{comp}_err"

            if comp in dft_slab.columns and err_col in dft_slab.columns:
                y_dft = dft_slab[comp].to_numpy()
                y_dft_err = dft_slab[err_col].to_numpy()
                ax.plot(
                    dft_slab["energy"],
                    y_dft, color=color, linewidth=1, label=comp)
                ax.fill_between(
                    dft_slab["energy"],
                    y_dft - y_dft_err,
                    y_dft + y_dft_err,
                    alpha=0.2,
                    color=color,
                )

            if comp in free_slab.columns and err_col in free_slab.columns:
                y_free = free_slab[comp].to_numpy()
                y_free_err = free_slab[err_col].to_numpy()
                ax.errorbar(
                    free_slab["energy"],
                    y_free,
                    yerr=y_free_err,
                    color=color,
                    marker="o",
                    linestyle="None",
                    markersize=1.5,
                    ecolor="k",
                    capsize=0.5,
                    elinewidth=0.5,
                )

        ax.set_ylabel(slab)
        ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

    axs[-1].set_xlabel("Energy (eV)")
    fig.align_labels()
    return axs

In [ ]:
axs = plot_n_traces(
    dft_n,
    free_n,
    components=["dichroism", "birefringence"]
)

for ax in axs:
    ax.legend()
    ax.set_xlim(279, 290)
plt.show()

In [ ]:
from scipy.optimize import Bounds, least_squares
from uncertainties import correlated_values


def _column_errors(slab_df, col, err_col_fmt):
    err_key = err_col_fmt.format(col=col)
    if err_key in slab_df.columns:
        err = slab_df[err_key].to_numpy(dtype=float)
    else:
        err = np.ones(slab_df[col].shape[0], dtype=float)
    return np.maximum(err, np.finfo(float).eps)


def fit_orientation_from_free(
    free_slab,
    ooc_df,
    dft_slab,
    energy_col="energy",
    xx_col="xx",
    zz_col="zz",
    ixx_col="ixx",
    izz_col="izz",
    err_col_fmt="{col}_err",
):
    """
  Fit slab orientation (gamma, radians) and density to free tensor components.

  Minimizes weighted squared residuals on ixx, izz, xx, and zz. Approximates
  parameter uncertainties from the residual Jacobian at the optimum and
  returns correlated ``ufloat`` values.

  Parameters
  ----------
  free_slab : pandas.DataFrame
      Per-slab free-model tensor components and optional error columns.
  ooc_df : pandas.DataFrame
      Oriented optical constants with columns ``energy``, ``n_ixx``, ``n_izz``,
      ``n_xx``, and ``n_zz``.
  dft_slab : pandas.DataFrame
      DFT slab row supplying initial ``rotation`` and ``density`` (as ``ufloat``).
  energy_col : str, optional
      Photon energy column in ``free_slab``.
  xx_col, zz_col, ixx_col, izz_col : str, optional
      Tensor component columns in ``free_slab``.
  err_col_fmt : str, optional
      Format string for error columns, e.g. ``"{col}_err"``.

  Returns
  -------
  gamma_fit : uncertainties.core.Variable
      Fitted tilt angle (radians) with standard uncertainty.
  density_fit : uncertainties.core.Variable
      Fitted density with standard uncertainty (correlated with ``gamma_fit``).
  result : scipy.optimize.OptimizeResult
      ``least_squares`` result including ``jac`` at the solution.
  cov : numpy.ndarray
      2x2 parameter covariance matrix, scaled by the residual mean square
      ``sum(residuals**2) / dof`` so uncertainties match the weighted fit scatter.
  """

    def model_components(gamma, density):
        energies = free_slab[energy_col].to_numpy()
        ixx = np.interp(energies, ooc_df["energy"], ooc_df["n_ixx"])
        izz = np.interp(energies, ooc_df["energy"], ooc_df["n_izz"])
        xx = np.interp(energies, ooc_df["energy"], ooc_df["n_xx"])
        zz = np.interp(energies, ooc_df["energy"], ooc_df["n_zz"])

        cos_gamma = np.cos(gamma)
        sin_gamma = np.sin(gamma)

        ixx_model = density * (ixx * (cos_gamma + 1) + izz * sin_gamma) / 2
        xx_model = density * (xx * (cos_gamma + 1) + zz * sin_gamma) / 2
        izz_model = density * (ixx * sin_gamma + izz * cos_gamma)
        zz_model = density * (xx * sin_gamma + zz * cos_gamma)

        return ixx_model, izz_model, xx_model, zz_model

    ixx = free_slab[ixx_col].to_numpy(dtype=float)
    izz = free_slab[izz_col].to_numpy(dtype=float)
    xx = free_slab[xx_col].to_numpy(dtype=float)
    zz = free_slab[zz_col].to_numpy(dtype=float)
    ixx_err = _column_errors(free_slab, ixx_col, err_col_fmt)
    izz_err = _column_errors(free_slab, izz_col, err_col_fmt)
    xx_err = _column_errors(free_slab, xx_col, err_col_fmt)
    zz_err = _column_errors(free_slab, zz_col, err_col_fmt)

    def residual_vector(params):
        gamma, density = params
        ixx_model, izz_model, xx_model, zz_model = model_components(gamma, density)
        return np.concatenate(
            [
                (ixx - ixx_model) / ixx_err,
                (izz - izz_model) / izz_err,
                (xx - xx_model) / xx_err,
                (zz - zz_model) / zz_err,
            ]
        )

    gamma0 = dft_slab["rotation"].values[0]
    density0 = dft_slab["density"].values[0]
    x0 = np.array([gamma0.nominal_value, density0.nominal_value], dtype=float)
    bounds = Bounds(
        lb=[
            min(0.0, gamma0.nominal_value - 5 * gamma0.std_dev),
            max(0.0, density0.nominal_value - density0.std_dev),
        ],
        ub=[
            min(np.pi / 2, gamma0.nominal_value + 5 * gamma0.std_dev),
            max(0.0, density0.nominal_value + density0.std_dev),
        ],
        keep_feasible=True,
    )

    result = least_squares(
        residual_vector,
        x0,
        bounds=(bounds.lb, bounds.ub),
        method="trf",
        jac="3-point",
    )

    jtj = result.jac.T @ result.jac
    try:
        cov_unit = np.linalg.inv(jtj)
    except np.linalg.LinAlgError:
        cov_unit = np.full((2, 2), np.nan)

    n_res = int(result.fun.size)
    n_par = int(result.x.size)
    dof = max(n_res - n_par, 1)
    mse = (2.0 * float(result.cost)) / dof
    cov = mse * cov_unit

    gamma_fit, density_fit = correlated_values(result.x, cov)
    return gamma_fit, density_fit, result, cov


slab_results = {}
for slab in free_n["slab"].unique():
    slab_df = free_n[free_n["slab"] == slab]
    dft_slab = dft_n[dft_n["slab"] == slab]
    gamma, density, result, cov = fit_orientation_from_free(slab_df, ooc, dft_slab)
    slab_results[slab] = {
        "gamma": gamma,
        "density": density,
        "result": result,
        "cov": cov,
    }


In [ ]:
import numpy as np
from refnx.reflect.interface import Erf, Step


def slab_depth_profile(thickness, roughness, orientation, dz=0.01, kernel_bound=5):
    """
    Depth profile with error-function-smoothed steps at each internal boundary.

    Parameters
    ----------
    thickness : sequence of float
        Thickness of every slab in order
        ``| vacuum | material_1 | ... | material_m | substrate |``.
        The vacuum entry is overwritten: it is not a physical thickness; see Notes.
    roughness : sequence of float
        Same length as ``thickness``. For ``k >= 1``, ``roughness[k]`` is the sigma
        (Angstrom) between slab ``k-1`` and slab ``k``. ``roughness[0]`` is ignored
        (vacuum has no roughness; the air-to-first-material step uses ``roughness[1]``).
    orientation : sequence of float
        Orientation of each material slab only, length ``len(thickness) - 2``.
        Vacuum and substrate use ``baseline`` internally.
    dz : float, optional
        Depth sampling step (Angstrom).
    kernel_bound : int, optional
        Depth grid extends past the outermost interfaces by this multiple of the
        adjacent interface sigma (minimum ``dz`` when sigma is zero).

    Returns
    -------
    z : np.ndarray
        Depth (Angstrom). First material surface at ``z == 0``; vacuum has ``z < 0``.
    gamma : np.ndarray
        Orientation at each ``z``.

    Notes
    -----
    With ``S = len(thickness)``, slab indices are ``0`` vacuum, ``1..S-2`` materials,
    ``S-1`` substrate. Vacuum thickness is replaced by at least
    ``kernel_bound * max(roughness[1], dz)`` Angstrom so the first interface is resolved.
    After replacement, interface between slabs ``i`` and ``i+1`` is at
    ``z = sum(thickness[1:i+1])`` and uses sigma ``roughness[i+1]``.
    Smoothed steps use ``refnx.reflect.interface.Erf``; zero sigma uses ``Step``.
    """
    baseline = np.arcsin(np.sqrt(2 / 3))
    t = np.asarray(thickness, dtype=float).copy()
    rho = np.asarray(roughness, dtype=float)
    s = int(t.size)
    if s < 2:
        raise ValueError("thickness must include at least vacuum and substrate slabs")
    if rho.shape != (s,):
        raise ValueError("roughness must have the same length as thickness")
    n_material = s - 2
    if len(orientation) != n_material:
        raise ValueError(
            "orientation must have length len(thickness) - 2 (material slabs only)"
        )
    sigma_vm = max(float(rho[1]), float(dz))
    t[0] = max(kernel_bound * sigma_vm, float(dz))
    interface_z = np.empty(s - 1, dtype=float)
    interface_z[0] = 0.0
    for i in range(1, s - 1):
        interface_z[i] = interface_z[i - 1] + t[i]
    sigma_if = rho[1:s]
    z_min = min(-t[0], -kernel_bound * max(float(sigma_if[0]), float(dz)))
    tail_sig = max(float(sigma_if[-1]), float(dz))
    z_max = float(interface_z[-1] + t[-1]) + kernel_bound * tail_sig
    z = np.arange(z_min, z_max, dz)
    zone_orient = np.concatenate(
        ([baseline], np.asarray(orientation, dtype=float), [baseline])
    )
    gamma = np.full_like(z, zone_orient[0])
    erf_f = Erf()
    step_f = Step()
    for i in range(s - 1):
        loc = interface_z[i]
        sig = float(sigma_if[i])
        delta = zone_orient[i + 1] - zone_orient[i]
        f = step_f if sig == 0 else erf_f
        scale = 0.0 if sig == 0 else sig
        gamma += delta * f(z, scale=scale, loc=loc)
    return z, gamma


def order_parameter(gamma):
    return (3 / 2) * (np.cos(gamma) ** 2 - 1 / 3)


def orientation_profile_uncertainty_gamma(
    thickness,
    roughness,
    gamma_means,
    gamma_errs,
    *,
    n_samples=4000,
    ci=0.68,
    dz=0.01,
    kernel_bound=5,
    random_state=0,
):
    """
    Propagate per-slab gamma uncertainty to depth profile and P2.

    Parameters
    ----------
    thickness, roughness : array_like
        Slab geometry as accepted by slab_depth_profile.
    gamma_means : sequence of float
        Nominal gamma per fitted slab, in radians, ordered to match the
        slab structure (vacuum and substrate baseline appended internally).
    gamma_errs : sequence of float
        1-sigma uncertainty on each entry of gamma_means, in radians.

    Returns
    -------
    z, gamma, gamma_lo, gamma_hi, p2, p2_lo, p2_hi : ndarrays
    """
    baseline = np.arcsin(np.sqrt(2 / 3))
    mus = np.asarray(gamma_means, dtype=float)
    sigmas = np.asarray(gamma_errs, dtype=float)
    if mus.shape != sigmas.shape:
        raise ValueError("gamma_means and gamma_errs must align")

    nominal_orient = mus.tolist() + [baseline]
    z, gamma = slab_depth_profile(
        thickness, roughness, nominal_orient, dz=dz, kernel_bound=kernel_bound
    )

    rng = np.random.default_rng(random_state)
    gamma_draws = np.empty((n_samples, z.size), dtype=float)
    alpha = (1.0 - ci) / 2.0

    for k in range(n_samples):
        sampled = np.where(
            sigmas > 0,
            rng.normal(mus, np.maximum(sigmas, 0.0)),
            mus,
        )
        sampled = np.clip(sampled, 0.0, np.pi / 2)
        orient = sampled.tolist() + [baseline]
        _, g_draw = slab_depth_profile(
            thickness, roughness, orient, dz=dz, kernel_bound=kernel_bound
        )
        gamma_draws[k] = g_draw

    gamma_lo = np.quantile(gamma_draws, alpha, axis=0)
    gamma_hi = np.quantile(gamma_draws, 1.0 - alpha, axis=0)

    p2_draws = (3.0 / 2.0) * (np.cos(gamma_draws) ** 2 - 1.0 / 3.0)
    p2 = order_parameter(gamma)
    p2_lo = np.quantile(p2_draws, alpha, axis=0)
    p2_hi = np.quantile(p2_draws, 1.0 - alpha, axis=0)

    return z, gamma, gamma_lo, gamma_hi, p2, p2_lo, p2_hi


def _gamma_bundle_from_dft_n(dft_n_df, slab_order):
    """
    Pull per-slab nominal rotation and its stderr from dft_n.

    extract_dft_optical_traces stores the ufloat in the rotation column and
    its s component in rotation_err. We grab the first row per slab because
    those values are constant across the energy axis.
    """
    means, errs = [], []
    for s in slab_order:
        row = dft_n_df[dft_n_df["slab"] == s].iloc[0]
        rot = row["rotation"]
        means.append(rot.n if hasattr(rot, "n") else float(rot))
        errs.append(float(row["rotation_err"]))
    return means, errs


In [ ]:
thicknesses = [
    s.thick.value for s in exp_constrained.objectives[0].model.structure
]
roughnesses = [
    s.rough.value for s in exp_constrained.objectives[0].model.structure
]

_isotropic_gamma = np.arcsin(np.sqrt(2 / 3))
_slab_order = list(free_n["slab"].unique())
orientations = [
    slab_results[s]["gamma"].nominal_value for s in _slab_order
] + [_isotropic_gamma]

for s in _slab_order:
    g = slab_results[s]["gamma"]
    print(
        f"{s}: gamma = {g.nominal_value:.4f} +/- {g.std_dev:.4f} rad "
        f"({np.degrees(g.std_dev):.2f} deg)"
    )

print(len(orientations), len(roughnesses), len(thicknesses))

In [ ]:
_slab_order_fig = list(free_n["slab"].unique())

free_gamma_means = [slab_results[s]["gamma"].nominal_value for s in _slab_order_fig]
free_gamma_errs = [slab_results[s]["gamma"].std_dev for s in _slab_order_fig]

(
    z,
    gamma,
    gamma_lo,
    gamma_hi,
    p2,
    p2_lo,
    p2_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses,
    roughnesses,
    free_gamma_means,
    free_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_deg = gamma * 180 / np.pi
gamma_deg_lo = gamma_lo * 180 / np.pi
gamma_deg_hi = gamma_hi * 180 / np.pi

print(
    "free profile gamma band (deg): "
    f"median={np.median(gamma_deg_hi - gamma_deg_lo):.3f}, "
    f"max={np.max(gamma_deg_hi - gamma_deg_lo):.3f}"
)
print(
    "profile P2 band: "
    f"median={np.median(p2_hi - p2_lo):.4f}, "
    f"max={np.max(p2_hi - p2_lo):.4f}"
)

In [ ]:
thicknesses_dft = [
    s.thick.value for s in dft_constrained.objectives[0].model.structure
]
roughnesses_dft = [
    s.rough.value for s in dft_constrained.objectives[0].model.structure
]
orientations_dft = [
    *[s.sld.rotation.value for s in dft_constrained.objectives[0].model.structure][1:-2],
    np.arcsin(np.sqrt(2 / 3)),
]

print(len(orientations_dft), len(roughnesses_dft), len(thicknesses_dft))

_slab_order_dft_fig = list(dft_n["slab"].unique())
dft_gamma_means, dft_gamma_errs = _gamma_bundle_from_dft_n(dft_n, _slab_order_dft_fig)

(
    z_dft_u,
    gamma_dft_u,
    gamma_dft_lo,
    gamma_dft_hi,
    p2_dft_u,
    p2_dft_lo,
    p2_dft_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses_dft,
    roughnesses_dft,
    dft_gamma_means,
    dft_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_dft_deg_u = gamma_dft_u * 180 / np.pi
gamma_dft_deg_lo = gamma_dft_lo * 180 / np.pi
gamma_dft_deg_hi = gamma_dft_hi * 180 / np.pi

z_dft, gamma_dft = slab_depth_profile(
    thicknesses_dft, roughnesses_dft, orientations_dft
)
gamma_dft_deg = gamma_dft * 180 / np.pi
p2_dft = order_parameter(gamma_dft)

print(
    "dft profile gamma band (deg): "
    f"median={np.median(gamma_dft_deg_hi - gamma_dft_deg_lo):.3f}, "
    f"max={np.max(gamma_dft_deg_hi - gamma_dft_deg_lo):.3f}"
)

In [ ]:
from matplotlib.patches import Patch


In [ ]:
# save the orientation profiles as csv files
df_free = pd.DataFrame({"z": z, "gamma": gamma})
df_dft = pd.DataFrame({"z": z_dft, "gamma": gamma_dft})
df_free.to_csv("free_tensor_orientation.csv", index=False)
df_dft.to_csv("dft_slab_orientation.csv", index=False)

In [ ]:
import colorsys


def shade_layers_p2_mapped(ax, z_profile, p2_profile, region_edges, region_labels, LAYER_CFG):
    """
    Signed P₂ color mapping over the full orientation profile.

    Encoding
    --------
    P₂ < 0  (face-on)   →  purple,  saturation ∝ |P₂|
    P₂ > 0  (edge-on)   →  amber,   saturation ∝ |P₂|
    P₂ = 0  (isotropic) →  pure gray (saturation = 0, guaranteed)

    Boundaries
    ----------
    - Surface side (z < film_start): alpha fades linearly to 0 so the fill
      dissolves into the white vacuum background.
    - Substrate side (z > film_end): alpha is held at ALPHA_BASE so the fill
      meets the SiO2/Si region with a hard edge.

    Flat fills for vacuum, SiO2, and Si are drawn from LAYER_CFG as before.
    """
    import matplotlib.colors as mcolors

    HUE_FACE_ON = 0.78    # purple — flat-lying / face-on
    HUE_EDGE_ON = 0.08    # amber  — upright / edge-on
    LIGHTNESS   = 0.68    # fixed; lower = richer color at full saturation
    P2_MAX      = p2_profile.max()
    P2_MIN      = p2_profile.min()
    MAX_SAT     = 0.90    # maximum achievable saturation
    ALPHA_BASE  = 0.72    # alpha inside and at the substrate edge

    # Film boundaries from the slab structure
    film_start = region_edges[1]    # vacuum → film
    film_end   = region_edges[-2]   # film → SiO2

    # Fade zone: distance over which alpha ramps from 0 → ALPHA_BASE
    # on the vacuum side.  Use the surface roughness scale (~10 Å typical).
    FADE_WIDTH = 15.0   # Å — adjust to match your surface roughness

    # ── Build per-sample RGBA ────────────────────────────────────────────────

    rgba = np.zeros((len(z_profile), 4))

    for i, (z_val, p2_val) in enumerate(zip(z_profile, p2_profile)):
        # Saturation: strictly zero when P₂ == 0 by construction
        if p2_val <= 0.0:
            # Face-on branch: normalize against the negative extreme
            sat = np.clip(p2_val / P2_MIN, 0.0, 1.0) * MAX_SAT
            hue = HUE_FACE_ON
        else:
            # Edge-on branch: normalize against the positive extreme
            sat = np.clip(p2_val / P2_MAX, 0.0, 1.0) * MAX_SAT
            hue = HUE_EDGE_ON

        # At sat == 0, hls_to_rgb returns a gray regardless of hue — correct.
        rgb = colorsys.hls_to_rgb(hue, LIGHTNESS, sat)

        # Alpha: fade to transparent on the vacuum side of film_start,
        # hold flat everywhere else (including inside substrate region).
        if z_val < film_start:
            # Linear ramp: 0 at (film_start - FADE_WIDTH), ALPHA_BASE at film_start
            t = (z_val - (film_start - FADE_WIDTH)) / FADE_WIDTH
            alpha = float(np.clip(t, 0.0, 1.0)) * ALPHA_BASE
        else:
            alpha = ALPHA_BASE

        rgba[i] = (*rgb, alpha)

    # ── Draw as pcolormesh ───────────────────────────────────────────────────
    # Set ylim before calling this function so the mesh spans the axes correctly.

    y_lo, y_hi = ax.get_ylim()
    z_edges = np.concatenate(
        [[z_profile[0]],
         0.5 * (z_profile[:-1] + z_profile[1:]),
         [z_profile[-1]]]
    )
    mesh_y     = np.array([y_lo, y_hi])
    color_grid = rgba[np.newaxis, :, :]   # shape (1, N, 4)

    ax.pcolormesh(
        z_edges, mesh_y, color_grid,
        shading="flat",
        zorder=0,
        rasterized=True,    # avoids hairline gaps and large vector PDFs
    )

    # ── Flat fills for isotropic substrate / vacuum regions ─────────────────

    iso_labels = ("vacuum", "SiO2", "Si")
    for j, label in enumerate(region_labels):
        if label not in iso_labels:
            continue
        fc, alpha = LAYER_CFG[label]
        ax.axvspan(
            region_edges[j], region_edges[j + 1],
            facecolor=fc, edgecolor="none", alpha=alpha, zorder=0,
        )

    # ── Boundary marker lines ────────────────────────────────────────────────

    for xv in (region_edges[1], region_edges[-2]):
        ax.axvline(xv, color="0.50", lw=0.5, ls=":", zorder=1)

In [ ]:
"""
Three-panel PRL figure (3.35 in wide):
  (a)  γ vs depth
  (b)  P₂ vs depth
  (c)  AIC / BIC / χ² comparison — grouped bar chart

Color convention
  DFT slab    →  #5DA5DA  (sky blue),    solid line
  Free tensor →  #FAA43A  (warm orange), dashed line
"""


# ── Helpers ──────────────────────────────────────────────────────────────────

def slab_depth_profile(thickness, roughness, orientation, dz=0.01, kernel_bound=5):
    baseline = np.arcsin(np.sqrt(2 / 3))
    t   = np.asarray(thickness,   dtype=float).copy()
    rho = np.asarray(roughness,   dtype=float)
    s   = int(t.size)
    sigma_vm = max(float(rho[1]), float(dz))
    t[0] = max(kernel_bound * sigma_vm, float(dz))
    interface_z    = np.empty(s - 1, dtype=float)
    interface_z[0] = 0.0
    for i in range(1, s - 1):
        interface_z[i] = interface_z[i - 1] + t[i]
    sigma_if = rho[1:s]
    z_min    = min(-t[0], -kernel_bound * max(float(sigma_if[0]), float(dz)))
    tail_sig = max(float(sigma_if[-1]), float(dz))
    z_max    = float(interface_z[-1] + t[-1]) + kernel_bound * tail_sig
    z = np.arange(z_min, z_max, dz)
    zone_orient = np.concatenate(
        ([baseline], np.asarray(orientation, dtype=float), [baseline])
    )
    gamma  = np.full_like(z, zone_orient[0])
    erf_f  = Erf()
    step_f = Step()
    for i in range(s - 1):
        loc   = interface_z[i]
        sig   = float(sigma_if[i])
        delta = zone_orient[i + 1] - zone_orient[i]
        f     = step_f if sig == 0 else erf_f
        scale = 0.0    if sig == 0 else sig
        gamma += delta * f(z, scale=scale, loc=loc)
    return z, gamma


def order_parameter(gamma):
    return (3 / 2) * (np.cos(gamma) ** 2 - 1 / 3)


def orientation_profile_uncertainty_gamma(
    thickness,
    roughness,
    gamma_means,
    gamma_errs,
    *,
    n_samples=4000,
    ci=0.68,
    dz=0.01,
    kernel_bound=5,
    random_state=0,
):
    baseline = np.arcsin(np.sqrt(2 / 3))
    mus = np.asarray(gamma_means, dtype=float)
    sigmas = np.asarray(gamma_errs, dtype=float)
    if mus.shape != sigmas.shape:
        raise ValueError("gamma_means and gamma_errs must align")

    nominal_orient = mus.tolist() + [baseline]
    z, gamma = slab_depth_profile(
        thickness, roughness, nominal_orient, dz=dz, kernel_bound=kernel_bound
    )

    rng = np.random.default_rng(random_state)
    gamma_draws = np.empty((n_samples, z.size), dtype=float)
    alpha = (1.0 - ci) / 2.0

    for k in range(n_samples):
        sampled = np.where(
            sigmas > 0,
            rng.normal(mus, np.maximum(sigmas, 0.0)),
            mus,
        )
        sampled = np.clip(sampled, 0.0, np.pi / 2)
        orient = sampled.tolist() + [baseline]
        _, g_draw = slab_depth_profile(
            thickness, roughness, orient, dz=dz, kernel_bound=kernel_bound
        )
        gamma_draws[k] = g_draw

    gamma_lo = np.quantile(gamma_draws, alpha, axis=0)
    gamma_hi = np.quantile(gamma_draws, 1.0 - alpha, axis=0)

    p2_draws = (3.0 / 2.0) * (np.cos(gamma_draws) ** 2 - 1.0 / 3.0)
    p2 = order_parameter(gamma)
    p2_lo = np.quantile(p2_draws, alpha, axis=0)
    p2_hi = np.quantile(p2_draws, 1.0 - alpha, axis=0)

    return z, gamma, gamma_lo, gamma_hi, p2, p2_lo, p2_hi


def _gamma_bundle_from_dft_n(dft_n_df, slab_order):
    means, errs = [], []
    for s in slab_order:
        row = dft_n_df[dft_n_df["slab"] == s].iloc[0]
        rot = row["rotation"]
        means.append(rot.n if hasattr(rot, "n") else float(rot))
        errs.append(float(row["rotation_err"]))
    return means, errs

# ── Color palette ─────────────────────────────────────────────────────────────

PALETTE = ["C0", "C1", "C2", "C3", "C4", "C5"]
C_DFT  = PALETTE[0]
C_FREE = PALETTE[1]

LAYER_CFG = {
    "vacuum":    ("#9B9B9B", 0.12),
    "surface":   ("#C4B8C8", 0.35),
    "bulk":      ("#C4B8C8", 0.60),
    "interface": ("#C4B8C8", 0.35),
    "SiO2":      ("#8A8A8A", 0.0),
    "Si":        ("#8A8A8A", 0.0),
}
dashes = (3, 6)

MAGIC_ANGLE_DEG = np.arcsin(np.sqrt(2 / 3)) * 180 / np.pi   # ≈ 54.74°

# Shared layout knobs. Adjust in one place, all three panels respond.
YLABEL_X    = -.08    # axes-fraction x for both ylabel and panel letter
YTICK_PAD   = 1.5      # gap between spine and tick text (points)
PANEL_LBL_Y = 1.06     # axes-fraction y for the (a) (b) (c) markers

# ── Data ─────────────────────────────────────────────────────────────────────

_slab_order_fig = list(free_n["slab"].unique())
free_gamma_means = [slab_results[s]["gamma"].nominal_value for s in _slab_order_fig]
free_gamma_errs = [slab_results[s]["gamma"].std_dev for s in _slab_order_fig]

(
    z,
    gamma,
    gamma_lo,
    gamma_hi,
    p2,
    p2_lo,
    p2_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses,
    roughnesses,
    free_gamma_means,
    free_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

z_dft, gamma_dft = slab_depth_profile(
    thicknesses_dft, roughnesses_dft, orientations_dft
)

gamma_deg = gamma * 180 / np.pi
gamma_deg_lo = gamma_lo * 180 / np.pi
gamma_deg_hi = gamma_hi * 180 / np.pi
gamma_dft_deg = gamma_dft * 180 / np.pi
p2_dft = order_parameter(gamma_dft)

# ── Region geometry ──────────────────────────────────────────────────────────

region_labels = ["vacuum", "surface", "bulk", "interface", "SiO2", "Si"]
cum_phys      = np.cumsum(np.asarray(thicknesses[1:], dtype=float))
region_edges  = [float(z[0]), 0.0] + cum_phys.tolist()
region_edges[-1] = max(region_edges[-1], float(z[-1]))

# ── Statistics ────────────────────────────────────────────────────────────────

stats = {
    "DFT slab":    {
        "chi2": 7.5,
        "aic": 223.8,
        "bic":  824.3,
        "N": len(dft_constrained.varying_parameters())
        },
    "Free tensor": {
        "chi2": 1.7,
        "aic": 591.6,
        "bic": 2529.1,
        "N": len(exp_constrained.varying_parameters())
        },
}
# ── Figure layout ─────────────────────────────────────────────────────────────

FIG_W = 3.35
FIG_H = 4.70

fig = plt.figure(figsize=(FIG_W, FIG_H))

gs_profiles = gridspec.GridSpec(
    2, 1,
    figure=fig,
    height_ratios=[1.10, 0.90],
    hspace=0.1,
    top=0.90,
    bottom=0.42,
)

gs_stat = gridspec.GridSpec(
    1, 1,
    figure=fig,
    top=0.30,
    bottom=0.08,
)

ax_gamma = fig.add_subplot(gs_profiles[0])
ax_p2    = fig.add_subplot(gs_profiles[1], sharex=ax_gamma)
ax_stat  = fig.add_subplot(gs_stat[0])

# Helper that pins the ylabel x-position and stamps the panel letter at the
# same x, guaranteeing both sit on a single vertical column across panels.
def _place_panel_label(ax, letter, *, y=PANEL_LBL_Y):
    ax.yaxis.set_label_coords(YLABEL_X, 0.5)
    ax.text(
        YLABEL_X, y, letter,
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=10,
    )

# ── Panel (a): γ vs depth ─────────────────────────────────────────────────────

ax_gamma.set_ylabel(r"$\gamma$ (deg)")
ax_gamma.set_xlim(region_edges[0], z.max())
ax_gamma.set_ylim(50, 75)
shade_layers_p2_mapped(ax_gamma, z, p2, region_edges, region_labels, LAYER_CFG)
ax_gamma.fill_between(
    z, gamma_deg_lo, gamma_deg_hi,
    color=C_FREE, alpha=0.35, linewidth=0, zorder=2,
)
ax_gamma.plot(z,     gamma_deg,     color=C_FREE, ls="--", lw=2,
              zorder=4, label="Free slab", dashes=dashes)
ax_gamma.plot(z_dft, gamma_dft_deg, color=C_DFT,  ls="-",  lw=2,
              zorder=3, label="DFT slab")
ax_gamma.axhline(MAGIC_ANGLE_DEG, color="0.35", lw=0.6, ls="-", zorder=2)
ax_gamma.yaxis.set_major_locator(ticker.MultipleLocator(5))
ax_gamma.yaxis.set_minor_locator(ticker.MultipleLocator(2.5))
ax_gamma.tick_params(axis="y", pad=YTICK_PAD)
ax_gamma.tick_params(labelbottom=False)

for lbl in ax_gamma.get_yticklabels():
    lbl.set_horizontalalignment("right")

ax_gamma.legend(
    loc="lower center",
    ncol=2,
    handlelength=1.4,
    handleheight=0.8,
    columnspacing=0.8,
    handletextpad=0.4,
    borderpad=0.0,
    frameon=False,
)

# Layer name annotations.
anno_tf = mpl.transforms.blended_transform_factory(
    ax_gamma.transData, ax_gamma.transAxes
)
layer_anno = [
    ("surface",   (region_edges[1] + region_edges[2]) / 2),
    ("bulk",      (region_edges[2] + region_edges[3]) / 2),
    ("interface", (region_edges[3] + region_edges[4]) / 2),
]
for lbl, xc in layer_anno:
    ax_gamma.text(
        xc, 1.03, lbl,
        transform=anno_tf,
        ha="center", va="bottom",
        fontsize=7, color="0.38",
        clip_on=False,
    )

_place_panel_label(ax_gamma, "(a)", y=1.10)

# ── Panel (b): P₂ vs depth ───────────────────────────────────────────────────

ax_p2.set_ylabel(r"$P_2$")
ax_p2.set_xlabel(r"Depth ($\AA$)")
ax_p2.set_xlim(region_edges[0], z.max())
ax_p2.set_ylim(-0.5, 0.25)
shade_layers_p2_mapped(ax_p2, z, p2, region_edges, region_labels, LAYER_CFG)
ax_p2.fill_between(
    z, p2_lo, p2_hi,
    color=C_FREE, alpha=0.35, linewidth=0, zorder=2,
)
ax_p2.plot(z,     p2,     color=C_FREE, ls="--", lw=2, zorder=4, dashes=dashes)
ax_p2.plot(z_dft, p2_dft, color=C_DFT,  ls="-",  lw=2, zorder=3)
ax_p2.axhline(
    0,
    color=plt.rcParams["axes.edgecolor"],
    lw=plt.rcParams["axes.linewidth"],
    ls="-",
    zorder=2,
)

p2_ticks  = [-0.5, -0.25, 0.0, 0.25]
p2_labels = [r"$-1/2$", r"$-1/4$", r"$0$", r"$1/4$"]
ax_p2.yaxis.set_major_locator(ticker.FixedLocator(p2_ticks))
ax_p2.yaxis.set_major_formatter(ticker.FixedFormatter(p2_labels))
ax_p2.yaxis.set_minor_locator(ticker.MultipleLocator(0.125))
ax_p2.tick_params(axis="y", pad=YTICK_PAD)

for lbl in ax_p2.get_yticklabels():
    lbl.set_horizontalalignment("right")

_place_panel_label(ax_p2, "(b)")

# ── Panel (c): statistics bar chart ──────────────────────────────────────────

METRICS     = [r"$\chi^2_\nu$", "N", "AIC", "BIC"]
BAR_W       = 0.28
GROUP_GAP   = 0.82
INTRA_GAP   = 0.04

models     = list(stats.keys())
bar_colors = [C_DFT, C_FREE]
mkeys      = ["chi2", "N", "aic", "bic"]

gcentres = np.arange(len(METRICS)) * GROUP_GAP
offsets = np.linspace(
    -((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
    ((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
    len(models)
)

raw_vals = {mk: np.array([stats[m][mk] for m in models]) for mk in mkeys}
norms    = {mk: v.max() for mk, v in raw_vals.items()}

for gi, mk in enumerate(mkeys):
    for bi, (model, color) in enumerate(zip(models, bar_colors)):
        raw  = stats[model][mk]
        norm = raw / norms[mk]
        x    = gcentres[gi] + offsets[bi]

        ax_stat.bar(
            x, norm,
            width=BAR_W,
            color=color,
            alpha=0.90,
            linewidth=0,
            zorder=2,
        )

        fmt = f"{raw:.1f}" if mk == "chi2" else f"{raw:.0f}"
        ax_stat.text(
            x, norm + 0.025,
            fmt,
            ha="center", va="bottom",
            fontsize=8
        )

ax_stat.set_ylim(0, 1.18)
ax_stat.set_xticks(gcentres)
ax_stat.set_xticklabels(METRICS)
ax_stat.set_yticklabels([])
ax_stat.grid(False)
ax_stat.axhline(1.0, color="0.65", lw=0.5, ls="--", zorder=1)
ax_stat.set_ylabel(r"AIC / BIC / $\chi^2$")

_place_panel_label(ax_stat, "(c)")

# ── Final export ─────────────────────────────────────────────────────────────
#
# fig.align_ylabels is no longer needed because every ylabel x-coordinate is
# pinned by hand to YLABEL_X. Leaving it would be harmless but redundant.

plt.savefig(
    "orientation_profiles.png", format="png", dpi=300,
    bbox_inches="tight", transparent=True
)

plt.show()

In [ ]:
"""
Three-panel PRL figure (3.35 in wide), five-variant comparison.

variants
  v1  nominal traces only, region shading                   no errors
  v2  free and DFT error bands, region shading              both errors
  v3  free and DFT error bands, white bg, layer barriers    both errors
  v4  free error band only, region shading                  free only
  v5  free error band only, white bg, layer barriers        free only

Color convention
  DFT slab    →  C0 sky blue,    solid line
  Free tensor →  C1 warm orange, dashed line

Prerequisite for v2 and v3.
  DFT gamma uncertainties are taken from dft_n rotation and rotation_err via
  _gamma_bundle_from_dft_n.
"""

# ── Helpers ──────────────────────────────────────────────────────────────────

def slab_depth_profile(thickness, roughness, orientation, dz=0.01, kernel_bound=5):
    baseline = np.arcsin(np.sqrt(2 / 3))
    t   = np.asarray(thickness,   dtype=float).copy()
    rho = np.asarray(roughness,   dtype=float)
    s   = int(t.size)
    sigma_vm = max(float(rho[1]), float(dz))
    t[0] = max(kernel_bound * sigma_vm, float(dz))
    interface_z    = np.empty(s - 1, dtype=float)
    interface_z[0] = 0.0
    for i in range(1, s - 1):
        interface_z[i] = interface_z[i - 1] + t[i]
    sigma_if = rho[1:s]
    z_min    = min(-t[0], -kernel_bound * max(float(sigma_if[0]), float(dz)))
    tail_sig = max(float(sigma_if[-1]), float(dz))
    z_max    = float(interface_z[-1] + t[-1]) + kernel_bound * tail_sig
    z = np.arange(z_min, z_max, dz)
    zone_orient = np.concatenate(
        ([baseline], np.asarray(orientation, dtype=float), [baseline])
    )
    gamma  = np.full_like(z, zone_orient[0])
    erf_f  = Erf()
    step_f = Step()
    for i in range(s - 1):
        loc   = interface_z[i]
        sig   = float(sigma_if[i])
        delta = zone_orient[i + 1] - zone_orient[i]
        f     = step_f if sig == 0 else erf_f
        scale = 0.0    if sig == 0 else sig
        gamma += delta * f(z, scale=scale, loc=loc)
    return z, gamma


def order_parameter(gamma):
    return (3 / 2) * (np.cos(gamma) ** 2 - 1 / 3)


def orientation_profile_uncertainty_gamma(
    thickness,
    roughness,
    gamma_means,
    gamma_errs,
    *,
    n_samples=4000,
    ci=0.68,
    dz=0.01,
    kernel_bound=5,
    random_state=0,
):
    baseline = np.arcsin(np.sqrt(2 / 3))
    mus = np.asarray(gamma_means, dtype=float)
    sigmas = np.asarray(gamma_errs, dtype=float)
    if mus.shape != sigmas.shape:
        raise ValueError("gamma_means and gamma_errs must align")

    nominal_orient = mus.tolist() + [baseline]
    z, gamma = slab_depth_profile(
        thickness, roughness, nominal_orient, dz=dz, kernel_bound=kernel_bound
    )

    rng = np.random.default_rng(random_state)
    gamma_draws = np.empty((n_samples, z.size), dtype=float)
    alpha = (1.0 - ci) / 2.0

    for k in range(n_samples):
        sampled = np.where(
            sigmas > 0,
            rng.normal(mus, np.maximum(sigmas, 0.0)),
            mus,
        )
        sampled = np.clip(sampled, 0.0, np.pi / 2)
        orient = sampled.tolist() + [baseline]
        _, g_draw = slab_depth_profile(
            thickness, roughness, orient, dz=dz, kernel_bound=kernel_bound
        )
        gamma_draws[k] = g_draw

    gamma_lo = np.quantile(gamma_draws, alpha, axis=0)
    gamma_hi = np.quantile(gamma_draws, 1.0 - alpha, axis=0)

    p2_draws = (3.0 / 2.0) * (np.cos(gamma_draws) ** 2 - 1.0 / 3.0)
    p2 = order_parameter(gamma)
    p2_lo = np.quantile(p2_draws, alpha, axis=0)
    p2_hi = np.quantile(p2_draws, 1.0 - alpha, axis=0)

    return z, gamma, gamma_lo, gamma_hi, p2, p2_lo, p2_hi


def _gamma_bundle_from_dft_n(dft_n_df, slab_order):
    means, errs = [], []
    for s in slab_order:
        row = dft_n_df[dft_n_df["slab"] == s].iloc[0]
        rot = row["rotation"]
        means.append(rot.n if hasattr(rot, "n") else float(rot))
        errs.append(float(row["rotation_err"]))
    return means, errs

# ── Color palette and layer config ───────────────────────────────────────────

PALETTE = ["C0", "C1", "C2", "C3", "C4", "C5"]
C_DFT  = PALETTE[0]
C_FREE = PALETTE[1]

LAYER_CFG = {
    "vacuum":    ("#9B9B9B", 0.12),
    "surface":   ("#C4B8C8", 0.35),
    "bulk":      ("#C4B8C8", 0.60),
    "interface": ("#C4B8C8", 0.35),
    "SiO2":      ("#8A8A8A", 0.0),
    "Si":        ("#8A8A8A", 0.0),
}
dashes = (3, 6)

MAGIC_ANGLE_DEG = np.arcsin(np.sqrt(2 / 3)) * 180 / np.pi   # ≈ 54.74°

YLABEL_X    = -0.18
YTICK_PAD   = 1.5
PANEL_LBL_Y = 1.06

FIG_W = 3.35
FIG_H = 4.70

# ── Free-tensor uncertainty bundle ───────────────────────────────────────────

_slab_order_fig = list(free_n["slab"].unique())
free_gamma_means = [slab_results[s]["gamma"].nominal_value for s in _slab_order_fig]
free_gamma_errs = [slab_results[s]["gamma"].std_dev for s in _slab_order_fig]

(
    z,
    gamma,
    gamma_lo,
    gamma_hi,
    p2,
    p2_lo,
    p2_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses,
    roughnesses,
    free_gamma_means,
    free_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_deg = gamma * 180 / np.pi
gamma_deg_lo = gamma_lo * 180 / np.pi
gamma_deg_hi = gamma_hi * 180 / np.pi

_slab_order_dft_fig = list(dft_n["slab"].unique())
dft_gamma_means, dft_gamma_errs = _gamma_bundle_from_dft_n(dft_n, _slab_order_dft_fig)

(
    z_dft_u,
    gamma_dft_u,
    gamma_dft_lo,
    gamma_dft_hi,
    p2_dft_u,
    p2_dft_lo,
    p2_dft_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses_dft,
    roughnesses_dft,
    dft_gamma_means,
    dft_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_dft_deg_u = gamma_dft_u * 180 / np.pi
gamma_dft_deg_lo = gamma_dft_lo * 180 / np.pi
gamma_dft_deg_hi = gamma_dft_hi * 180 / np.pi

z_dft, gamma_dft = slab_depth_profile(
    thicknesses_dft, roughnesses_dft, orientations_dft
)
gamma_dft_deg = gamma_dft * 180 / np.pi
p2_dft = order_parameter(gamma_dft)

# ── Region geometry ──────────────────────────────────────────────────────────

region_labels = ["vacuum", "surface", "bulk", "interface", "SiO2", "Si"]
cum_phys      = np.cumsum(np.asarray(thicknesses[1:], dtype=float))
region_edges  = [float(z[0]), 0.0] + cum_phys.tolist()
region_edges[-1] = max(region_edges[-1], float(z[-1]))

# Interior layer boundaries (drop the outermost left and right plot edges).
LAYER_BARRIERS = region_edges[1:-1]

# ── Statistics ───────────────────────────────────────────────────────────────

stats = {
    "DFT slab":    {
        "chi2": 7.5, "aic": 223.8, "bic": 824.3,
        "N": len(dft_constrained.varying_parameters()),
    },
    "Free tensor": {
        "chi2": 1.7, "aic": 591.6, "bic": 2529.1,
        "N": len(exp_constrained.varying_parameters()),
    },
}

# ── Figure factory ───────────────────────────────────────────────────────────

def make_orientation_figure(
    *,
    show_free_errors,
    show_dft_errors,
    show_shading,
    show_barriers,
    savename,
):
    """Render one variant of the three-panel orientation figure."""

    fig = plt.figure(figsize=(FIG_W, FIG_H))

    gs_profiles = gridspec.GridSpec(
        2, 1, figure=fig,
        height_ratios=[1.10, 0.90],
        hspace=0.1, top=0.90, bottom=0.42,
    )
    gs_stat = gridspec.GridSpec(
        1, 1, figure=fig, top=0.30, bottom=0.08,
    )

    ax_gamma = fig.add_subplot(gs_profiles[0])
    ax_p2    = fig.add_subplot(gs_profiles[1], sharex=ax_gamma)
    ax_stat  = fig.add_subplot(gs_stat[0])

    def _place_panel_label(ax, letter, *, y=PANEL_LBL_Y):
        ax.yaxis.set_label_coords(YLABEL_X, 0.5)
        ax.text(
            YLABEL_X, y, letter,
            transform=ax.transAxes,
            ha="right", va="bottom",
            fontsize=10,
        )

    def _draw_barriers(ax):
        for xi in LAYER_BARRIERS:
            ax.axvline(xi, color="k", lw=1, ls="--", zorder=5)

    # ── Panel (a): γ vs depth ────────────────────────────────────────────────

    ax_gamma.set_ylabel(r"$\gamma$ (deg)")
    ax_gamma.set_xlim(region_edges[0], z.max())
    ax_gamma.set_ylim(50, 75)

    if show_shading:
        shade_layers_p2_mapped(
            ax_gamma, z, p2, region_edges, region_labels, LAYER_CFG
        )
    if show_barriers:
        _draw_barriers(ax_gamma)

    if show_free_errors:
        ax_gamma.fill_between(
            z, gamma_deg_lo, gamma_deg_hi,
            color=C_FREE, alpha=0.30, linewidth=0, zorder=2,
        )
    if show_dft_errors:
        ax_gamma.fill_between(
            z_dft_u, gamma_dft_deg_lo, gamma_dft_deg_hi,
            color=C_DFT, alpha=0.30, linewidth=0, zorder=2,
        )

    ax_gamma.plot(
        z, gamma_deg, color=C_FREE, ls="--", lw=2,
        zorder=4, label="Free slab", dashes=dashes,
    )
    ax_gamma.plot(
        z_dft, gamma_dft_deg, color=C_DFT, ls="-", lw=2,
        zorder=3, label="DFT slab",
    )
    ax_gamma.axhline(MAGIC_ANGLE_DEG, color="0.35", lw=0.6, ls="-", zorder=2)
    ax_gamma.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_gamma.yaxis.set_minor_locator(ticker.MultipleLocator(2.5))
    ax_gamma.tick_params(axis="y", pad=YTICK_PAD)
    ax_gamma.tick_params(labelbottom=False)

    for lbl in ax_gamma.get_yticklabels():
        lbl.set_horizontalalignment("right")

    leg = ax_gamma.legend(
        loc="lower center", ncol=2,
        handlelength=2, handleheight=0.8,
        columnspacing=0.8, handletextpad=0.4,
        borderpad=0.0, frameon=False,
    )
    # Set legend handle for "Free slab" to regular dashed line
    if leg:
        for lh in leg.get_lines():
            if lh.get_label() == "Free slab":
                lh.set_linestyle('--')
                lh.set_dashes((6, 6))  # regular dashed pattern


    anno_tf = mpl.transforms.blended_transform_factory(
        ax_gamma.transData, ax_gamma.transAxes
    )
    layer_anno = [
        ("surface",   (region_edges[1] + region_edges[2]) / 2),
        ("bulk",      (region_edges[2] + region_edges[3]) / 2),
        ("interface", (region_edges[3] + region_edges[4]) / 2),
    ]
    for lbl, xc in layer_anno:
        ax_gamma.text(
            xc, 1.03, lbl,
            transform=anno_tf,
            ha="center", va="bottom",
            fontsize=7, color="0.38",
            clip_on=False,
        )

    _place_panel_label(ax_gamma, "(a)", y=1.10)

    # ── Panel (b): P₂ vs depth ───────────────────────────────────────────────

    ax_p2.set_ylabel(r"$P_2$")
    ax_p2.set_xlabel(r"Depth ($\AA$)")
    ax_p2.set_xlim(region_edges[0], z.max())
    ax_p2.set_ylim(-0.5, 0.25)

    if show_shading:
        shade_layers_p2_mapped(
            ax_p2, z, p2, region_edges, region_labels, LAYER_CFG
        )
    if show_barriers:
        _draw_barriers(ax_p2)

    if show_free_errors:
        ax_p2.fill_between(
            z, p2_lo, p2_hi,
            color=C_FREE, alpha=0.30, linewidth=0, zorder=2,
        )
    if show_dft_errors:
        ax_p2.fill_between(
            z_dft_u, p2_dft_lo, p2_dft_hi,
            color=C_DFT, alpha=0.30, linewidth=0, zorder=2,
        )

    ax_p2.plot(
        z, p2, color=C_FREE, ls="--", lw=2,
        zorder=4, dashes=dashes,
    )
    ax_p2.plot(
        z_dft, p2_dft, color=C_DFT, ls="-", lw=2, zorder=3,
    )
    ax_p2.axhline(
        0,
        color=plt.rcParams["axes.edgecolor"],
        lw=plt.rcParams["axes.linewidth"],
        ls="-", zorder=2,
    )

    p2_ticks  = [-0.5, -0.25, 0.0, 0.25]
    p2_labels = [r"$-1/2$", r"$-1/4$", r"$0$", r"$1/4$"]
    ax_p2.yaxis.set_major_locator(ticker.FixedLocator(p2_ticks))
    ax_p2.yaxis.set_major_formatter(ticker.FixedFormatter(p2_labels))
    ax_p2.yaxis.set_minor_locator(ticker.MultipleLocator(0.125))
    ax_p2.tick_params(axis="y", pad=YTICK_PAD)

    for lbl in ax_p2.get_yticklabels():
        lbl.set_horizontalalignment("right")

    _place_panel_label(ax_p2, "(b)")

    # ── Panel (c): statistics bar chart ─────────────────────────────────────

    METRICS   = [r"$\chi^2_\nu$", "N", "AIC", "BIC"]
    BAR_W     = 0.28
    GROUP_GAP = 0.82
    INTRA_GAP = 0.04

    models     = list(stats.keys())
    bar_colors = [C_DFT, C_FREE]
    mkeys      = ["chi2", "N", "aic", "bic"]

    gcentres = np.arange(len(METRICS)) * GROUP_GAP
    offsets = np.linspace(
        -((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
        ((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
        len(models),
    )

    raw_vals = {mk: np.array([stats[m][mk] for m in models]) for mk in mkeys}
    norms    = {mk: v.max() for mk, v in raw_vals.items()}

    for gi, mk in enumerate(mkeys):
        for bi, (model, color) in enumerate(zip(models, bar_colors)):
            raw  = stats[model][mk]
            norm = raw / norms[mk]
            x    = gcentres[gi] + offsets[bi]
            ax_stat.bar(
                x, norm, width=BAR_W, color=color,
                alpha=0.90, linewidth=0, zorder=2,
            )
            fmt = f"{raw:.1f}" if mk == "chi2" else f"{raw:.0f}"
            ax_stat.text(
                x, norm + 0.025, fmt,
                ha="center", va="bottom", fontsize=8,
            )

    ax_stat.set_ylim(0, 1.18)
    ax_stat.set_xticks(gcentres)
    ax_stat.set_xticklabels(METRICS)
    ax_stat.set_yticklabels([])
    ax_stat.grid(False)
    ax_stat.axhline(1.0, color="0.65", lw=0.5, ls="--", zorder=1)
    ax_stat.set_ylabel(r"AIC / BIC / $\chi^2$")

    _place_panel_label(ax_stat, "(c)")

    plt.savefig(
        savename, format="png", dpi=600,
        bbox_inches="tight", transparent=True,
    )
    plt.show()


# v1  nominal traces only, region shading
make_orientation_figure(
    show_free_errors=False,
    show_dft_errors=False,
    show_shading=True,
    show_barriers=False,
    savename="fig3_orientation.png",
)

In [ ]:
"""
Three-panel PRL figure (3.35 in wide), five-variant comparison.

variants
  v1  nominal traces only, region shading                   no errors
  v2  free and DFT error bands, region shading              both errors
  v3  free and DFT error bands, white bg, layer barriers    both errors
  v4  free error band only, region shading                  free only
  v5  free error band only, white bg, layer barriers        free only

Panels
  (a)  gamma vs depth
  (b)  ZnPc mass density vs depth (replaces P2)
  (c)  AIC / BIC / chi2 / N comparison, down arrow marks "lower is better"

Color convention
  DFT slab    ->  C0, solid line
  Free tensor ->  C1, dashed line

Prerequisites
  slab_results          free-fit gamma and density ufloats per slab
  dft_n                 DFT bundle with rotation, rotation_err, density ufloats
  thicknesses(_dft), roughnesses(_dft), orientations_dft
  dft_constrained, exp_constrained (GlobalObjective, for panel c statistics)
"""

import colorsys

from matplotlib.lines import Line2D

# ── Helpers ──────────────────────────────────────────────────────────────────

def slab_depth_profile(thickness, roughness, orientation, dz=0.01, kernel_bound=5):
    baseline = np.arcsin(np.sqrt(2 / 3))
    zone = [baseline, *list(orientation), baseline]
    return value_depth_profile(
        thickness, roughness, zone, dz=dz, kernel_bound=kernel_bound
    )


def value_depth_profile(thickness, roughness, zone_values, dz=0.01, kernel_bound=5):
    """
    Erf-smoothed depth profile of an arbitrary per-slab scalar.

    zone_values has one entry per slab in ``thickness`` (vacuum and substrate
    endpoints included). Geometry handling is identical to the original
    slab_depth_profile; the orientation profile and the density profile are
    both special cases of this kernel.
    """
    t    = np.asarray(thickness, dtype=float).copy()
    rho  = np.asarray(roughness, dtype=float)
    vals = np.asarray(zone_values, dtype=float)
    s    = int(t.size)
    if vals.shape != (s,):
        raise ValueError("zone_values must have one entry per slab")
    sigma_vm = max(float(rho[1]), float(dz))
    t[0] = max(kernel_bound * sigma_vm, float(dz))
    interface_z    = np.empty(s - 1, dtype=float)
    interface_z[0] = 0.0
    for i in range(1, s - 1):
        interface_z[i] = interface_z[i - 1] + t[i]
    sigma_if = rho[1:s]
    z_min    = min(-t[0], -kernel_bound * max(float(sigma_if[0]), float(dz)))
    tail_sig = max(float(sigma_if[-1]), float(dz))
    z_max    = float(interface_z[-1] + t[-1]) + kernel_bound * tail_sig
    z = np.arange(z_min, z_max, dz)
    prof   = np.full_like(z, vals[0])
    erf_f  = Erf()
    step_f = Step()
    for i in range(s - 1):
        loc   = interface_z[i]
        sig   = float(sigma_if[i])
        delta = vals[i + 1] - vals[i]
        f     = step_f if sig == 0 else erf_f
        scale = 0.0    if sig == 0 else sig
        prof += delta * f(z, scale=scale, loc=loc)
    return z, prof


def order_parameter(gamma):
    return (3 / 2) * (np.cos(gamma) ** 2 - 1 / 3)


def _rho_zones(thickness, rho_film, rho_substrate):
    """
    Map film and substrate densities onto the full slab stack.

    Vacuum carries zero density; the film slabs carry the fitted ZnPc
    densities; the trailing slabs (SiO2, Si) carry the substrate mass
    densities pulled from their MaterialSLD parameters so the profile
    steps up through the buried interface rather than decaying to zero.
    """
    zones = [0.0, *list(rho_film), *list(rho_substrate)]
    if len(zones) != len(thickness):
        raise ValueError(
            f"zone count {len(zones)} does not match slab count "
            f"{len(thickness)}; check rho_film / rho_substrate lengths"
        )
    return zones


def _substrate_densities(structure, n_expected):
    """
    Mass densities (g cm^-3) of the substrate MaterialSLD slabs.

    The fronting medium is itself a MaterialSLD (Vacuum_*), so the first
    component is skipped on position rather than name; the remaining
    MaterialSLD slabs are the Oxide and Substrate. Raises if the count
    disagrees with the slab bookkeeping so a silent mismatch cannot shift
    the profile.
    """
    components = list(getattr(structure, "components", structure))
    rho, names = [], []
    for s in components[1:]:   # skip fronting (vacuum)
        sld = getattr(s, "sld", None)
        if sld is None or not isinstance(sld, fit.MaterialSLD):
            continue
        d = sld.density
        rho.append(float(d.value) if hasattr(d, "value") else float(d))
        names.append(s.name)
    if len(rho) != n_expected:
        raise ValueError(
            f"found {len(rho)} substrate MaterialSLD slabs {names}, "
            f"expected {n_expected}"
        )
    return rho


def orientation_profile_uncertainty_gamma(
    thickness,
    roughness,
    gamma_means,
    gamma_errs,
    *,
    n_samples=4000,
    ci=0.68,
    dz=0.01,
    kernel_bound=5,
    random_state=0,
):
    baseline = np.arcsin(np.sqrt(2 / 3))
    mus = np.asarray(gamma_means, dtype=float)
    sigmas = np.asarray(gamma_errs, dtype=float)
    if mus.shape != sigmas.shape:
        raise ValueError("gamma_means and gamma_errs must align")

    nominal_orient = mus.tolist() + [baseline]
    z, gamma = slab_depth_profile(
        thickness, roughness, nominal_orient, dz=dz, kernel_bound=kernel_bound
    )

    rng = np.random.default_rng(random_state)
    gamma_draws = np.empty((n_samples, z.size), dtype=float)
    alpha = (1.0 - ci) / 2.0

    for k in range(n_samples):
        sampled = np.where(
            sigmas > 0,
            rng.normal(mus, np.maximum(sigmas, 0.0)),
            mus,
        )
        sampled = np.clip(sampled, 0.0, np.pi / 2)
        orient = sampled.tolist() + [baseline]
        _, g_draw = slab_depth_profile(
            thickness, roughness, orient, dz=dz, kernel_bound=kernel_bound
        )
        gamma_draws[k] = g_draw

    gamma_lo = np.quantile(gamma_draws, alpha, axis=0)
    gamma_hi = np.quantile(gamma_draws, 1.0 - alpha, axis=0)

    p2_draws = (3.0 / 2.0) * (np.cos(gamma_draws) ** 2 - 1.0 / 3.0)
    p2 = order_parameter(gamma)
    p2_lo = np.quantile(p2_draws, alpha, axis=0)
    p2_hi = np.quantile(p2_draws, 1.0 - alpha, axis=0)

    return z, gamma, gamma_lo, gamma_hi, p2, p2_lo, p2_hi


def density_profile_uncertainty(
    thickness,
    roughness,
    rho_means,
    rho_errs,
    rho_substrate,
    *,
    n_samples=4000,
    ci=0.68,
    dz=0.01,
    kernel_bound=5,
    random_state=0,
):
    """
    Propagate per-slab film density uncertainty to the depth profile.

    Same Monte Carlo scheme as the gamma version. Film draws are clipped at
    zero since the density scale factor is non-negative by construction; the
    substrate densities are held fixed across draws.
    """
    mus = np.asarray(rho_means, dtype=float)
    sigmas = np.asarray(rho_errs, dtype=float)
    if mus.shape != sigmas.shape:
        raise ValueError("rho_means and rho_errs must align")

    z, rho_prof = value_depth_profile(
        thickness, roughness, _rho_zones(thickness, mus, rho_substrate),
        dz=dz, kernel_bound=kernel_bound,
    )

    rng = np.random.default_rng(random_state)
    draws = np.empty((n_samples, z.size), dtype=float)
    alpha = (1.0 - ci) / 2.0

    for k in range(n_samples):
        sampled = np.where(
            sigmas > 0,
            rng.normal(mus, np.maximum(sigmas, 0.0)),
            mus,
        )
        sampled = np.clip(sampled, 0.0, None)
        _, d_draw = value_depth_profile(
            thickness, roughness, _rho_zones(thickness, sampled, rho_substrate),
            dz=dz, kernel_bound=kernel_bound,
        )
        draws[k] = d_draw

    rho_lo = np.quantile(draws, alpha, axis=0)
    rho_hi = np.quantile(draws, 1.0 - alpha, axis=0)
    return z, rho_prof, rho_lo, rho_hi


def _gamma_bundle_from_dft_n(dft_n_df, slab_order):
    means, errs = [], []
    for s in slab_order:
        row = dft_n_df[dft_n_df["slab"] == s].iloc[0]
        rot = row["rotation"]
        means.append(rot.n if hasattr(rot, "n") else float(rot))
        errs.append(float(row["rotation_err"]))
    return means, errs


def _density_bundle_from_dft_n(dft_n_df, slab_order):
    means, errs = [], []
    for s in slab_order:
        row = dft_n_df[dft_n_df["slab"] == s].iloc[0]
        d = row["density"]
        means.append(d.n if hasattr(d, "n") else float(d))
        errs.append(d.s if hasattr(d, "s") else 0.0)
    return means, errs


def shade_layers_p2_density(
    ax, z_profile, p2_profile, rho_profile, region_edges, region_labels, LAYER_CFG
):
    """
    Signed P2 color mapping with mass-density-weighted opacity.

    Encoding
    --------
    hue         P2 < 0 -> purple (face-on), P2 > 0 -> amber (edge-on)
    saturation  proportional to |P2| against its profile extreme
    alpha       ALPHA_BASE * rho(z) / max(rho), against the full mass
                density profile. Vacuum is fully transparent, the film
                sits at rho_film / rho_Si of the base alpha, and the Si
                substrate is darkest. P2 -> 0 in the substrate, so the
                dark region renders as neutral gray rather than carrying
                orientation color.
    """
    HUE_FACE_ON = 0.78
    HUE_EDGE_ON = 0.08
    LIGHTNESS   = 0.68
    P2_MAX      = p2_profile.max()
    P2_MIN      = p2_profile.min()
    MAX_SAT     = 0.90
    ALPHA_BASE  = 0.72

    rho_w = np.clip(
        np.asarray(rho_profile, dtype=float) / float(np.max(rho_profile)),
        0.0, 1.0,
    )

    rgba = np.zeros((len(z_profile), 4))
    for i, (p2_val, w) in enumerate(zip(p2_profile, rho_w)):
        if p2_val <= 0.0:
            sat = np.clip(p2_val / P2_MIN, 0.0, 1.0) * MAX_SAT
            hue = HUE_FACE_ON
        else:
            sat = np.clip(p2_val / P2_MAX, 0.0, 1.0) * MAX_SAT
            hue = HUE_EDGE_ON
        rgb = colorsys.hls_to_rgb(hue, LIGHTNESS, sat)
        rgba[i] = (*rgb, ALPHA_BASE * w)

    y_lo, y_hi = ax.get_ylim()
    z_edges = np.concatenate(
        [[z_profile[0]],
         0.5 * (z_profile[:-1] + z_profile[1:]),
         [z_profile[-1]]]
    )
    ax.pcolormesh(
        z_edges, np.array([y_lo, y_hi]), rgba[np.newaxis, :, :],
        shading="flat", zorder=0, rasterized=True,
    )

    iso_labels = ("vacuum", "SiO2", "Si")
    for j, label in enumerate(region_labels):
        if label not in iso_labels:
            continue
        fc, alpha = LAYER_CFG[label]
        ax.axvspan(
            region_edges[j], region_edges[j + 1],
            facecolor=fc, edgecolor="none", alpha=alpha, zorder=0,
        )

    for xv in (region_edges[1], region_edges[-2]):
        ax.axvline(xv, color="0.50", lw=0.5, ls=":", zorder=1)


def model_selection_stats(objective, *, use_logl=False):
    """
    Reduced chi2, AIC, and BIC from a (Global)Objective.

    The chi2 key is always the residual-based reduced chi-square,
    sum(r^2) / (n - k), independent of the flag.

    AIC and BIC come in two conventions that differ by a constant. For
    Gaussian errors, -2 lnL = chi2 + sum_i ln(2 pi sigma_i^2). The second
    term depends only on the data uncertainties, so it is identical for
    both models and cancels in dAIC and dBIC.

    use_logl=False  chi-square forms, AIC = chi2 + 2k, BIC = chi2 + k ln n.
                    The absolute values carry the (large) total chi2.
    use_logl=True   likelihood forms, AIC = 2k - 2 lnL, BIC = k ln n - 2 lnL
                    from objective.logl(). This is the standard definition
                    and what panel (c) should display.

    The two must agree on dAIC and dBIC; if they do not, the per-point
    error terms differ between the models and the comparison is invalid.
    """
    resid = np.asarray(objective.residuals(), dtype=float)
    n = int(resid.size)
    k = len(objective.varying_parameters())
    chi2 = float(np.sum(resid**2))
    nu = max(n - k, 1)
    if use_logl:
        logl = float(objective.logl())
        aic = 2.0 * k - 2.0 * logl
        bic = k * np.log(n) - 2.0 * logl
    else:
        aic = chi2 + 2.0 * k
        bic = chi2 + k * np.log(n)
    return {
        "chi2": chi2 / nu,
        "aic": aic,
        "bic": bic,
        "N": k,
        "npoints": n,
    }


def log_model_weights(stats_dict, key):
    """
    Log-space relative model weights from information criterion differences.

    ln w_i = -Delta_i / 2 - ln sum_j exp(-Delta_j / 2), evaluated with
    logaddexp so Deltas in the hundreds or thousands neither overflow nor
    underflow. For key="aic" the w_i are Akaike weights (probability of
    being the K-L-best model in the candidate set); for key="bic" they are
    posterior model probabilities under equal priors, with
    exp(-dBIC / 2) the BIC estimate of the Bayes factor.

    Returns
    -------
    ln_w : dict
        Natural log of each model's weight.
    delta : dict
        Criterion differences from the minimum.
    """
    names = list(stats_dict)
    vals = np.array([float(stats_dict[m][key]) for m in names])
    delta = vals - vals.min()
    ln_unnorm = -0.5 * delta
    ln_w = ln_unnorm - np.logaddexp.reduce(ln_unnorm)
    return dict(zip(names, ln_w)), dict(zip(names, delta))

# ── Color palette and layer config ───────────────────────────────────────────

PALETTE = ["C0", "C1", "C2", "C3", "C4", "C5"]
C_DFT  = PALETTE[0]
C_FREE = PALETTE[1]

LAYER_CFG = {
    "vacuum":    ("white", 0.12),
    "surface":   ("#C4B8C8", 0.35),
    "bulk":      ("#C4B8C8", 0.60),
    "interface": ("#C4B8C8", 0.35),
    "SiO2":      ("grey", 0.0),
    "Si":        ("black", 0.0),
}
dashes = (3, 6)

MAGIC_ANGLE_DEG = np.arcsin(np.sqrt(2 / 3)) * 180 / np.pi   # ≈ 54.74°

YLABEL_X    = -0.115
YTICK_PAD   = 1.5
PANEL_LBL_Y = 1.06

FIG_W = 3.35
FIG_H = 4.70

# ── Free-tensor uncertainty bundle ───────────────────────────────────────────

_slab_order_fig = list(free_n["slab"].unique())
free_gamma_means = [slab_results[s]["gamma"].nominal_value for s in _slab_order_fig]
free_gamma_errs = [slab_results[s]["gamma"].std_dev for s in _slab_order_fig]

(
    z,
    gamma,
    gamma_lo,
    gamma_hi,
    p2,
    p2_lo,
    p2_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses,
    roughnesses,
    free_gamma_means,
    free_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_deg = gamma * 180 / np.pi
gamma_deg_lo = gamma_lo * 180 / np.pi
gamma_deg_hi = gamma_hi * 180 / np.pi

_slab_order_dft_fig = list(dft_n["slab"].unique())
dft_gamma_means, dft_gamma_errs = _gamma_bundle_from_dft_n(dft_n, _slab_order_dft_fig)

(
    z_dft_u,
    gamma_dft_u,
    gamma_dft_lo,
    gamma_dft_hi,
    p2_dft_u,
    p2_dft_lo,
    p2_dft_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses_dft,
    roughnesses_dft,
    dft_gamma_means,
    dft_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_dft_deg_u = gamma_dft_u * 180 / np.pi
gamma_dft_deg_lo = gamma_dft_lo * 180 / np.pi
gamma_dft_deg_hi = gamma_dft_hi * 180 / np.pi

z_dft, gamma_dft = slab_depth_profile(
    thicknesses_dft, roughnesses_dft, orientations_dft
)
gamma_dft_deg = gamma_dft * 180 / np.pi

# ── Density profile bundles ──────────────────────────────────────────────────
# Free film densities come from the per-slab least-squares fit (slab_results);
# DFT film densities from the UniTensorSLD parameters carried in dft_n.
# Substrate (SiO2, Si) mass densities are read off the MaterialSLD slabs so
# the panel (b) profile steps up through the buried interface. All are
# erf-smoothed through the same thickness/roughness geometry as gamma.

free_rho_means = [slab_results[s]["density"].nominal_value for s in _slab_order_fig]
free_rho_errs  = [slab_results[s]["density"].std_dev for s in _slab_order_fig]

rho_sub_free = _substrate_densities(
    exp_constrained.objectives[0].model.structure,
    len(thicknesses) - len(free_rho_means) - 1,
)

(
    z_rho,
    rho_free,
    rho_free_lo,
    rho_free_hi,
) = density_profile_uncertainty(
    thicknesses,
    roughnesses,
    free_rho_means,
    free_rho_errs,
    rho_sub_free,
    n_samples=4000,
    ci=0.68,
)

dft_rho_means, dft_rho_errs = _density_bundle_from_dft_n(dft_n, _slab_order_dft_fig)

rho_sub_dft = _substrate_densities(
    dft_constrained.objectives[0].model.structure,
    len(thicknesses_dft) - len(dft_rho_means) - 1,
)

(
    z_rho_dft,
    rho_dft,
    rho_dft_lo,
    rho_dft_hi,
) = density_profile_uncertainty(
    thicknesses_dft,
    roughnesses_dft,
    dft_rho_means,
    dft_rho_errs,
    rho_sub_dft,
    n_samples=4000,
    ci=0.68,
)

# Shading alpha channel follows the full mass-density profile (film and
# substrate), normalized to its maximum, so opacity tracks where matter
# actually is: transparent vacuum, intermediate film, darkest Si substrate.
rho_on_z = np.interp(z, z_rho, rho_free)

# ── Region geometry ──────────────────────────────────────────────────────────

region_labels = ["vacuum", "surface", "bulk", "interface", "SiO2", "Si"]
cum_phys      = np.cumsum(np.asarray(thicknesses[1:], dtype=float))
region_edges  = [float(z[0]), 0.0] + cum_phys.tolist()
region_edges[-1] = max(region_edges[-1], float(z[-1]))

# Interior layer boundaries (drop the outermost left and right plot edges).
LAYER_BARRIERS = region_edges[1:-1]

# ── Statistics ───────────────────────────────────────────────────────────────

# Panel (c) displays the likelihood-form criteria (STATS_USE_LOGL = True).
# The chi-square forms differ by the model-independent constant
# sum_i ln(2 pi sigma_i^2), which inflates the absolute scale without
# moving dAIC or dBIC; the sanity block below verifies that agreement on
# every run. The displayed chi2_nu is residual-based in both conventions.
STATS_USE_LOGL = True

stats = {
    "DFT slab":    model_selection_stats(dft_constrained, use_logl=STATS_USE_LOGL),
    "Free tensor": model_selection_stats(exp_constrained, use_logl=STATS_USE_LOGL),
}

# Sanity check: both conventions side by side; the deltas must agree.
_stats_chi2 = {
    "DFT slab":    model_selection_stats(dft_constrained, use_logl=False),
    "Free tensor": model_selection_stats(exp_constrained, use_logl=False),
}
_stats_logl = {
    "DFT slab":    model_selection_stats(dft_constrained, use_logl=True),
    "Free tensor": model_selection_stats(exp_constrained, use_logl=True),
}
for _m in stats:
    _c, _l = _stats_chi2[_m], _stats_logl[_m]
    print(
        f"{_m}: n={_c['npoints']}, k={_c['N']}, chi2_nu={_c['chi2']:.2f} | "
        f"chi2-form AIC={_c['aic']:.1f}, BIC={_c['bic']:.1f} | "
        f"logl-form AIC={_l['aic']:.1f}, BIC={_l['bic']:.1f}"
    )
print(
    "dAIC agreement (chi2-form vs logl-form): "
    f"{_stats_chi2['Free tensor']['aic'] - _stats_chi2['DFT slab']['aic']:.2f} vs "
    f"{_stats_logl['Free tensor']['aic'] - _stats_logl['DFT slab']['aic']:.2f}"
)

# Akaike weights and Bayes factor, evaluated in log space. The pairwise
# ratio exp(-d/2) over- or underflows past |d| ~ 1400, so everything is
# carried as ln and reported as log10. ln BF(DFT/Free) = -dBIC/2 is the
# BIC (Schwarz) estimate of the Bayes factor under equal model priors;
# the analogous AIC quantity is the relative likelihood / evidence ratio.
_LN10 = float(np.log(10.0))
for _key, _label, _ratio_name in (
    ("aic", "Akaike", "evidence ratio"),
    ("bic", "Bayes (BIC)", "Bayes factor"),
):
    _ln_w, _ = log_model_weights(stats, key=_key)
    _ln_ratio = -0.5 * (stats["DFT slab"][_key] - stats["Free tensor"][_key])
    print(
        f"{_label}: d{_key.upper()}(DFT - Free) = "
        f"{stats['DFT slab'][_key] - stats['Free tensor'][_key]:+.1f} | "
        f"log10 {_ratio_name}(DFT/Free) = {_ln_ratio / _LN10:+.2f} | "
        f"P(DFT) = {np.exp(_ln_w['DFT slab']):.6f}, "
        f"log10 P(DFT) = {_ln_w['DFT slab'] / _LN10:.4g}, "
        f"log10 P(Free) = {_ln_w['Free tensor'] / _LN10:.4g}"
    )

# Experimental correction (non-structural) parameter counts for panel (c).
#   Both models: 21 s-pol + 21 p-pol scale factors and
#                21 s-pol + 21 p-pol theta offsets.
#   DFT additionally carries one global energy offset.
N_CORR = {
    "DFT slab":    21 * 4 + 1,
    "Free tensor": 21 * 4,
}

# ── Figure factory ───────────────────────────────────────────────────────────

def make_orientation_figure(
    *,
    show_free_errors,
    show_dft_errors,
    show_shading,
    show_barriers,
    savename,
):
    """Render one variant of the three-panel orientation figure."""

    fig = plt.figure(figsize=(FIG_W, FIG_H))

    gs_profiles = gridspec.GridSpec(
        2, 1, figure=fig,
        height_ratios=[1.10, 0.90],
        hspace=0.1, top=0.90, bottom=0.42,
    )
    gs_stat = gridspec.GridSpec(
        1, 1, figure=fig, top=0.30, bottom=0.08,
    )

    ax_gamma = fig.add_subplot(gs_profiles[0])
    ax_rho   = fig.add_subplot(gs_profiles[1], sharex=ax_gamma)
    ax_stat  = fig.add_subplot(gs_stat[0])

    def _place_panel_label(ax, letter, *, y=PANEL_LBL_Y):
        ax.yaxis.set_label_coords(YLABEL_X, 0.5)
        ax.text(
            YLABEL_X, y, letter,
            transform=ax.transAxes,
            ha="right", va="bottom",
            fontsize=10,
        )

    def _draw_barriers(ax):
        for xi in LAYER_BARRIERS:
            ax.axvline(xi, color="grey", lw=1, ls="--", zorder=5)

    # ── Panel (a): γ vs depth ────────────────────────────────────────────────

    ax_gamma.set_ylabel(r"$\gamma$ (deg)")
    ax_gamma.set_xlim(region_edges[0], z.max())
    ax_gamma.set_ylim(50, 75)

    if show_shading:
        shade_layers_p2_density(
            ax_gamma, z, p2, rho_on_z, region_edges, region_labels, LAYER_CFG
        )
    if show_barriers:
        _draw_barriers(ax_gamma)

    if show_free_errors:
        ax_gamma.fill_between(
            z, gamma_deg_lo, gamma_deg_hi,
            color=C_FREE, alpha=0.30, linewidth=0, zorder=2,
        )
    if show_dft_errors:
        ax_gamma.fill_between(
            z_dft_u, gamma_dft_deg_lo, gamma_dft_deg_hi,
            color=C_DFT, alpha=0.30, linewidth=0, zorder=2,
        )

    ax_gamma.plot(
        z, gamma_deg, color=C_FREE, ls="--", lw=2,
        zorder=4, dashes=dashes,
    )
    ax_gamma.plot(
        z_dft, gamma_dft_deg, color=C_DFT, ls="-", lw=2,
        zorder=3,
    )
    ax_gamma.axhline(MAGIC_ANGLE_DEG, color="0.35", lw=0.6, ls="-", zorder=2)
    ax_gamma.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_gamma.yaxis.set_minor_locator(ticker.MultipleLocator(2.5))
    ax_gamma.tick_params(axis="y", pad=YTICK_PAD)
    ax_gamma.tick_params(labelbottom=False)

    for lbl in ax_gamma.get_yticklabels():
        lbl.set_horizontalalignment("right")

    # Proxy handles. The on-plot free trace uses sparse (3, 6) dashes so the
    # DFT trace shows through, but at handlelength ~2 that pattern collapses
    # to a single segment and reads as solid. The legend therefore carries
    # its own dense dash pattern that is unambiguous at handle scale.
    legend_handles = [
        Line2D(
            [], [], color=C_FREE, lw=2,
            ls=(0, (2.8, 1.6)), label="Free slab",
        ),
        Line2D(
            [], [], color=C_DFT, lw=2,
            ls="-", label="DFT slab",
        ),
    ]
    ax_gamma.legend(
        handles=legend_handles,
        loc="lower center", ncol=2,
        handlelength=2.8, handleheight=0.8,
        columnspacing=0.8, handletextpad=0.4,
        borderpad=0.0, frameon=False,
    )

    anno_tf = mpl.transforms.blended_transform_factory(
        ax_gamma.transData, ax_gamma.transAxes
    )
    layer_anno = [
        ("surface",   (region_edges[1] + region_edges[2]) / 2),
        ("bulk",      (region_edges[2] + region_edges[3]) / 2),
        ("interface", (region_edges[3] + region_edges[4]) / 2),
    ]
    for lbl, xc in layer_anno:
        ax_gamma.text(
            xc, 1.03, lbl,
            transform=anno_tf,
            ha="center", va="bottom",
            fontsize=7, color="0.38",
            clip_on=False,
        )

    _place_panel_label(ax_gamma, "(a)", y=1.10)

    # ── Panel (b): ρ vs depth ────────────────────────────────────────────────

    ax_rho.set_ylabel(r"$\rho$ (g cm$^{-3}$)")
    ax_rho.set_xlabel(r"Depth ($\AA$)")
    ax_rho.set_xlim(-10, z.max())
    rho_top = 1.15 * max(float(rho_free.max()), float(rho_dft.max()))
    ax_rho.set_ylim(0, rho_top)

    if show_shading:
        shade_layers_p2_density(
            ax_rho, z, p2, rho_on_z, region_edges, region_labels, LAYER_CFG
        )
    if show_barriers:
        _draw_barriers(ax_rho)

    if show_free_errors:
        ax_rho.fill_between(
            z_rho, rho_free_lo, rho_free_hi,
            color=C_FREE, alpha=0.30, linewidth=0, zorder=2,
        )
    if show_dft_errors:
        ax_rho.fill_between(
            z_rho_dft, rho_dft_lo, rho_dft_hi,
            color=C_DFT, alpha=0.30, linewidth=0, zorder=2,
        )

    ax_rho.plot(
        z_rho, rho_free, color=C_FREE, ls="--", lw=2,
        zorder=4, dashes=dashes,
    )
    ax_rho.plot(
        z_rho_dft, rho_dft, color=C_DFT, ls="-", lw=2, zorder=3,
    )

    ax_rho.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_rho.yaxis.set_minor_locator(ticker.MultipleLocator(0.25))
    ax_rho.tick_params(axis="y", pad=YTICK_PAD)

    for lbl in ax_rho.get_yticklabels():
        lbl.set_horizontalalignment("right")

    _place_panel_label(ax_rho, "(b)")

    # ── Panel (c): statistics bar chart ─────────────────────────────────────

    METRICS   = [r"$\chi^2_\nu$", "N", "AIC", "BIC"]
    BAR_W     = 0.28
    GROUP_GAP = 0.82
    INTRA_GAP = 0.04

    models     = list(stats.keys())
    bar_colors = [C_DFT, C_FREE]
    mkeys      = ["chi2", "N", "aic", "bic"]

    gcentres = np.arange(len(METRICS)) * GROUP_GAP
    offsets = np.linspace(
        -((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
        ((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
        len(models),
    )

    raw_vals = {mk: np.array([stats[m][mk] for m in models]) for mk in mkeys}
    norms    = {mk: v.max() for mk, v in raw_vals.items()}

    for gi, mk in enumerate(mkeys):
        for bi, (model, color) in enumerate(zip(models, bar_colors)):
            raw  = stats[model][mk]
            norm = raw / norms[mk]
            x    = gcentres[gi] + offsets[bi]
            if mk == "N":
                # Experimental correction floor (per-energy scale factors
                # and theta offsets, plus the DFT energy offset), hatched;
                # structural parameters stack solid on top.
                corr = N_CORR[model] / norms[mk]
                ax_stat.bar(
                    x, corr, width=BAR_W, color=color,
                    alpha=0.90, linewidth=0, zorder=2,
                    hatch="/////", edgecolor="white",
                )
                ax_stat.bar(
                    x, norm - corr, bottom=corr, width=BAR_W, color=color,
                    alpha=0.90, linewidth=0, zorder=2,
                )
            else:
                ax_stat.bar(
                    x, norm, width=BAR_W, color=color,
                    alpha=0.90, linewidth=0, zorder=2,
                )
            fmt = f"{raw:.1f}" if mk == "chi2" else f"{raw:.0f}"
            ax_stat.text(
                x, norm + 0.025, fmt,
                ha="center", va="bottom", fontsize=8,
            )

    ax_stat.set_ylim(0, 1.18)
    ax_stat.set_xticks(gcentres)
    ax_stat.set_xticklabels(METRICS)
    ax_stat.set_yticklabels([])
    ax_stat.grid(False)
    ax_stat.axhline(1.0, color="0.65", lw=0.5, ls="--", zorder=1)
    ax_stat.set_ylabel(r"AIC / BIC / $\chi^2$")

    # Down arrow just outside the y spine. Lower is better for all model
    # selection criteria shown. annotation_clip=False keeps it through
    # bbox_inches="tight".
    ax_stat.annotate(
        "",
        xy=(-0.045, 0.25), xytext=(-0.045, 0.92),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(
            arrowstyle="-|>", color="0.30", lw=0.9,
            shrinkA=0, shrinkB=0, mutation_scale=8,
        ),
        annotation_clip=False,
    )
    ax_stat.text(
        -0.068, 0.585, "better",
        transform=ax_stat.transAxes,
        rotation=90, ha="center", va="center",
        fontsize=6.5, color="0.30",
    )

    _place_panel_label(ax_stat, "(c)")

    plt.savefig(
        savename, format="png", dpi=600,
        bbox_inches="tight", transparent=True,
    )
    plt.show()


# v1  nominal traces only, region shading
make_orientation_figure(
    show_free_errors=False,
    show_dft_errors=False,
    show_shading=True,
    show_barriers=False,
    savename="fig3_orientation.png",
)

In [ ]:
"""
Three-panel PRL figure (3.35 in wide), five-variant comparison.

variants
  v1  nominal traces only, region shading                   no errors
  v2  free and DFT error bands, region shading              both errors
  v3  free and DFT error bands, white bg, layer barriers    both errors
  v4  free error band only, region shading                  free only
  v5  free error band only, white bg, layer barriers        free only

Panels
  (a)  gamma vs depth
  (b)  ZnPc mass density vs depth (replaces P2)
  (c)  AIC / BIC / chi2 / N comparison, down arrow marks "lower is better"

Color convention
  DFT slab    ->  C0, solid line
  Free tensor ->  C1, dashed line

Prerequisites
  slab_results          free-fit gamma and density ufloats per slab
  dft_n                 DFT bundle with rotation, rotation_err, density ufloats
  thicknesses(_dft), roughnesses(_dft), orientations_dft
  dft_constrained, exp_constrained (GlobalObjective, for panel c statistics)
"""

import colorsys

import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

# White under-stroke that keeps small annotations legible on any background.
STROKE = [pe.Stroke(linewidth=1.5, foreground="white"), pe.Normal()]

# ── Helpers ──────────────────────────────────────────────────────────────────

def slab_depth_profile(thickness, roughness, orientation, dz=0.01, kernel_bound=5):
    baseline = np.arcsin(np.sqrt(2 / 3))
    zone = [baseline, *list(orientation), baseline]
    return value_depth_profile(
        thickness, roughness, zone, dz=dz, kernel_bound=kernel_bound
    )


def value_depth_profile(thickness, roughness, zone_values, dz=0.01, kernel_bound=5):
    """
    Erf-smoothed depth profile of an arbitrary per-slab scalar.

    zone_values has one entry per slab in ``thickness`` (vacuum and substrate
    endpoints included). Geometry handling is identical to the original
    slab_depth_profile; the orientation profile and the density profile are
    both special cases of this kernel.
    """
    t    = np.asarray(thickness, dtype=float).copy()
    rho  = np.asarray(roughness, dtype=float)
    vals = np.asarray(zone_values, dtype=float)
    s    = int(t.size)
    if vals.shape != (s,):
        raise ValueError("zone_values must have one entry per slab")
    sigma_vm = max(float(rho[1]), float(dz))
    t[0] = max(kernel_bound * sigma_vm, float(dz))
    interface_z    = np.empty(s - 1, dtype=float)
    interface_z[0] = 0.0
    for i in range(1, s - 1):
        interface_z[i] = interface_z[i - 1] + t[i]
    sigma_if = rho[1:s]
    z_min    = min(-t[0], -kernel_bound * max(float(sigma_if[0]), float(dz)))
    tail_sig = max(float(sigma_if[-1]), float(dz))
    z_max    = float(interface_z[-1] + t[-1]) + kernel_bound * tail_sig
    z = np.arange(z_min, z_max, dz)
    prof   = np.full_like(z, vals[0])
    erf_f  = Erf()
    step_f = Step()
    for i in range(s - 1):
        loc   = interface_z[i]
        sig   = float(sigma_if[i])
        delta = vals[i + 1] - vals[i]
        f     = step_f if sig == 0 else erf_f
        scale = 0.0    if sig == 0 else sig
        prof += delta * f(z, scale=scale, loc=loc)
    return z, prof


def order_parameter(gamma):
    return (3 / 2) * (np.cos(gamma) ** 2 - 1 / 3)


def _rho_zones(thickness, rho_film, rho_substrate):
    """
    Map film and substrate densities onto the full slab stack.

    Vacuum carries zero density; the film slabs carry the fitted ZnPc
    densities; the trailing slabs (SiO2, Si) carry the substrate mass
    densities pulled from their MaterialSLD parameters so the profile
    steps up through the buried interface rather than decaying to zero.
    """
    zones = [0.0, *list(rho_film), *list(rho_substrate)]
    if len(zones) != len(thickness):
        raise ValueError(
            f"zone count {len(zones)} does not match slab count "
            f"{len(thickness)}; check rho_film / rho_substrate lengths"
        )
    return zones


def _substrate_densities(structure, n_expected):
    """
    Mass densities (g cm^-3) of the substrate MaterialSLD slabs.

    The fronting medium is itself a MaterialSLD (Vacuum_*), so the first
    component is skipped on position rather than name; the remaining
    MaterialSLD slabs are the Oxide and Substrate. Raises if the count
    disagrees with the slab bookkeeping so a silent mismatch cannot shift
    the profile.
    """
    components = list(getattr(structure, "components", structure))
    rho, names = [], []
    for s in components[1:]:   # skip fronting (vacuum)
        sld = getattr(s, "sld", None)
        if sld is None or not isinstance(sld, fit.MaterialSLD):
            continue
        d = sld.density
        rho.append(float(d.value) if hasattr(d, "value") else float(d))
        names.append(s.name)
    if len(rho) != n_expected:
        raise ValueError(
            f"found {len(rho)} substrate MaterialSLD slabs {names}, "
            f"expected {n_expected}"
        )
    return rho


def orientation_profile_uncertainty_gamma(
    thickness,
    roughness,
    gamma_means,
    gamma_errs,
    *,
    n_samples=4000,
    ci=0.68,
    dz=0.01,
    kernel_bound=5,
    random_state=0,
):
    baseline = np.arcsin(np.sqrt(2 / 3))
    mus = np.asarray(gamma_means, dtype=float)
    sigmas = np.asarray(gamma_errs, dtype=float)
    if mus.shape != sigmas.shape:
        raise ValueError("gamma_means and gamma_errs must align")

    nominal_orient = mus.tolist() + [baseline]
    z, gamma = slab_depth_profile(
        thickness, roughness, nominal_orient, dz=dz, kernel_bound=kernel_bound
    )

    rng = np.random.default_rng(random_state)
    gamma_draws = np.empty((n_samples, z.size), dtype=float)
    alpha = (1.0 - ci) / 2.0

    for k in range(n_samples):
        sampled = np.where(
            sigmas > 0,
            rng.normal(mus, np.maximum(sigmas, 0.0)),
            mus,
        )
        sampled = np.clip(sampled, 0.0, np.pi / 2)
        orient = sampled.tolist() + [baseline]
        _, g_draw = slab_depth_profile(
            thickness, roughness, orient, dz=dz, kernel_bound=kernel_bound
        )
        gamma_draws[k] = g_draw

    gamma_lo = np.quantile(gamma_draws, alpha, axis=0)
    gamma_hi = np.quantile(gamma_draws, 1.0 - alpha, axis=0)

    p2_draws = (3.0 / 2.0) * (np.cos(gamma_draws) ** 2 - 1.0 / 3.0)
    p2 = order_parameter(gamma)
    p2_lo = np.quantile(p2_draws, alpha, axis=0)
    p2_hi = np.quantile(p2_draws, 1.0 - alpha, axis=0)

    return z, gamma, gamma_lo, gamma_hi, p2, p2_lo, p2_hi


def density_profile_uncertainty(
    thickness,
    roughness,
    rho_means,
    rho_errs,
    rho_substrate,
    *,
    n_samples=4000,
    ci=0.68,
    dz=0.01,
    kernel_bound=5,
    random_state=0,
):
    """
    Propagate per-slab film density uncertainty to the depth profile.

    Same Monte Carlo scheme as the gamma version. Film draws are clipped at
    zero since the density scale factor is non-negative by construction; the
    substrate densities are held fixed across draws.
    """
    mus = np.asarray(rho_means, dtype=float)
    sigmas = np.asarray(rho_errs, dtype=float)
    if mus.shape != sigmas.shape:
        raise ValueError("rho_means and rho_errs must align")

    z, rho_prof = value_depth_profile(
        thickness, roughness, _rho_zones(thickness, mus, rho_substrate),
        dz=dz, kernel_bound=kernel_bound,
    )

    rng = np.random.default_rng(random_state)
    draws = np.empty((n_samples, z.size), dtype=float)
    alpha = (1.0 - ci) / 2.0

    for k in range(n_samples):
        sampled = np.where(
            sigmas > 0,
            rng.normal(mus, np.maximum(sigmas, 0.0)),
            mus,
        )
        sampled = np.clip(sampled, 0.0, None)
        _, d_draw = value_depth_profile(
            thickness, roughness, _rho_zones(thickness, sampled, rho_substrate),
            dz=dz, kernel_bound=kernel_bound,
        )
        draws[k] = d_draw

    rho_lo = np.quantile(draws, alpha, axis=0)
    rho_hi = np.quantile(draws, 1.0 - alpha, axis=0)
    return z, rho_prof, rho_lo, rho_hi


def _gamma_bundle_from_dft_n(dft_n_df, slab_order):
    means, errs = [], []
    for s in slab_order:
        row = dft_n_df[dft_n_df["slab"] == s].iloc[0]
        rot = row["rotation"]
        means.append(rot.n if hasattr(rot, "n") else float(rot))
        errs.append(float(row["rotation_err"]))
    return means, errs


def _density_bundle_from_dft_n(dft_n_df, slab_order):
    means, errs = [], []
    for s in slab_order:
        row = dft_n_df[dft_n_df["slab"] == s].iloc[0]
        d = row["density"]
        means.append(d.n if hasattr(d, "n") else float(d))
        errs.append(d.s if hasattr(d, "s") else 0.0)
    return means, errs


def shade_layers_p2_density(
    ax, z_profile, p2_profile, rho_profile, region_edges, region_labels, LAYER_CFG
):
    """
    Signed P2 color mapping with mass-density-weighted opacity.

    Encoding
    --------
    hue         P2 < 0 -> purple (face-on), P2 > 0 -> amber (edge-on)
    saturation  proportional to |P2| against its profile extreme
    alpha       ALPHA_BASE * rho(z) / max(rho), against the full mass
                density profile. Vacuum is fully transparent, the film
                sits at rho_film / rho_Si of the base alpha, and the Si
                substrate is darkest. P2 -> 0 in the substrate, so the
                dark region renders as neutral gray rather than carrying
                orientation color.

    Region boundaries are drawn at every interior slab edge by default,
    as paired phase-offset white and gray dashes so they remain visible
    over the purple film, the dark substrate, and the pale vacuum alike.
    """
    HUE_FACE_ON = 0.78
    HUE_EDGE_ON = 0.08
    LIGHTNESS   = 0.68
    P2_MAX      = p2_profile.max()
    P2_MIN      = p2_profile.min()
    MAX_SAT     = 0.90
    ALPHA_BASE  = 0.72

    rho_w = np.clip(
        np.asarray(rho_profile, dtype=float) / float(np.max(rho_profile)),
        0.0, 1.0,
    )

    rgba = np.zeros((len(z_profile), 4))
    for i, (p2_val, w) in enumerate(zip(p2_profile, rho_w)):
        if p2_val <= 0.0:
            sat = np.clip(p2_val / P2_MIN, 0.0, 1.0) * MAX_SAT
            hue = HUE_FACE_ON
        else:
            sat = np.clip(p2_val / P2_MAX, 0.0, 1.0) * MAX_SAT
            hue = HUE_EDGE_ON
        rgb = colorsys.hls_to_rgb(hue, LIGHTNESS, sat)
        rgba[i] = (*rgb, ALPHA_BASE * w)

    y_lo, y_hi = ax.get_ylim()
    z_edges = np.concatenate(
        [[z_profile[0]],
         0.5 * (z_profile[:-1] + z_profile[1:]),
         [z_profile[-1]]]
    )
    ax.pcolormesh(
        z_edges, np.array([y_lo, y_hi]), rgba[np.newaxis, :, :],
        shading="flat", zorder=0, rasterized=True,
    )

    iso_labels = ("vacuum", "SiO2", "Si")
    for j, label in enumerate(region_labels):
        if label not in iso_labels:
            continue
        fc, alpha = LAYER_CFG[label]
        ax.axvspan(
            region_edges[j], region_edges[j + 1],
            facecolor=fc, edgecolor="none", alpha=alpha, zorder=0,
        )

    # Region boundaries, drawn by default at every interior edge.
    for xv in region_edges[1:-1]:
        ax.axvline(xv, color="white", lw=0.6, ls=(0, (2.5, 2.5)), zorder=1)
        ax.axvline(xv, color="0.40", lw=0.6, ls=(2.5, (2.5, 2.5)), zorder=1)


def model_selection_stats(objective, *, use_logl=False):
    """
    Reduced chi2, AIC, and BIC from a (Global)Objective.

    The chi2 key is always the residual-based reduced chi-square,
    sum(r^2) / (n - k), independent of the flag.

    AIC and BIC come in two conventions that differ by a constant. For
    Gaussian errors, -2 lnL = chi2 + sum_i ln(2 pi sigma_i^2). The second
    term depends only on the data uncertainties, so it is identical for
    both models and cancels in dAIC and dBIC.

    use_logl=False  chi-square forms, AIC = chi2 + 2k, BIC = chi2 + k ln n.
                    The absolute values carry the (large) total chi2.
    use_logl=True   likelihood forms, AIC = 2k - 2 lnL, BIC = k ln n - 2 lnL
                    from objective.logl(). This is the standard definition
                    and what panel (c) should display.

    The two must agree on dAIC and dBIC; if they do not, the per-point
    error terms differ between the models and the comparison is invalid.
    """
    resid = np.asarray(objective.residuals(), dtype=float)
    n = int(resid.size)
    k = len(objective.varying_parameters())
    chi2 = float(np.sum(resid**2))
    nu = max(n - k, 1)
    if use_logl:
        logl = float(objective.logl())
        aic = 2.0 * k - 2.0 * logl
        bic = k * np.log(n) - 2.0 * logl
    else:
        aic = chi2 + 2.0 * k
        bic = chi2 + k * np.log(n)
    return {
        "chi2": chi2 / nu,
        "aic": aic,
        "bic": bic,
        "N": k,
        "npoints": n,
    }


def log_model_weights(stats_dict, key):
    """
    Log-space relative model weights from information criterion differences.

    ln w_i = -Delta_i / 2 - ln sum_j exp(-Delta_j / 2), evaluated with
    logaddexp so Deltas in the hundreds or thousands neither overflow nor
    underflow. For key="aic" the w_i are Akaike weights (probability of
    being the K-L-best model in the candidate set); for key="bic" they are
    posterior model probabilities under equal priors, with
    exp(-dBIC / 2) the BIC estimate of the Bayes factor.

    Returns
    -------
    ln_w : dict
        Natural log of each model's weight.
    delta : dict
        Criterion differences from the minimum.
    """
    names = list(stats_dict)
    vals = np.array([float(stats_dict[m][key]) for m in names])
    delta = vals - vals.min()
    ln_unnorm = -0.5 * delta
    ln_w = ln_unnorm - np.logaddexp.reduce(ln_unnorm)
    return dict(zip(names, ln_w)), dict(zip(names, delta))

# ── Color palette and layer config ───────────────────────────────────────────

PALETTE = ["C0", "C1", "C2", "C3", "C4", "C5"]
C_DFT  = PALETTE[0]
C_FREE = PALETTE[1]

LAYER_CFG = {
    "vacuum":    ("white", 0.12),
    "surface":   ("#C4B8C8", 0.35),
    "bulk":      ("#C4B8C8", 0.60),
    "interface": ("#C4B8C8", 0.35),
    "SiO2":      ("grey", 0.0),
    "Si":        ("black", 0.0),
}
dashes = (3, 6)

MAGIC_ANGLE_DEG = np.arcsin(np.sqrt(2 / 3)) * 180 / np.pi   # ≈ 54.74°

YLABEL_X    = -0.115
YTICK_PAD   = 1.5
PANEL_LBL_Y = 1.06

FIG_W = 3.35
FIG_H = 4.70

# ── Free-tensor uncertainty bundle ───────────────────────────────────────────

_slab_order_fig = list(free_n["slab"].unique())
free_gamma_means = [slab_results[s]["gamma"].nominal_value for s in _slab_order_fig]
free_gamma_errs = [slab_results[s]["gamma"].std_dev for s in _slab_order_fig]

(
    z,
    gamma,
    gamma_lo,
    gamma_hi,
    p2,
    p2_lo,
    p2_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses,
    roughnesses,
    free_gamma_means,
    free_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_deg = gamma * 180 / np.pi
gamma_deg_lo = gamma_lo * 180 / np.pi
gamma_deg_hi = gamma_hi * 180 / np.pi

_slab_order_dft_fig = list(dft_n["slab"].unique())
dft_gamma_means, dft_gamma_errs = _gamma_bundle_from_dft_n(dft_n, _slab_order_dft_fig)

(
    z_dft_u,
    gamma_dft_u,
    gamma_dft_lo,
    gamma_dft_hi,
    p2_dft_u,
    p2_dft_lo,
    p2_dft_hi,
) = orientation_profile_uncertainty_gamma(
    thicknesses_dft,
    roughnesses_dft,
    dft_gamma_means,
    dft_gamma_errs,
    n_samples=4000,
    ci=0.68,
)

gamma_dft_deg_u = gamma_dft_u * 180 / np.pi
gamma_dft_deg_lo = gamma_dft_lo * 180 / np.pi
gamma_dft_deg_hi = gamma_dft_hi * 180 / np.pi

z_dft, gamma_dft = slab_depth_profile(
    thicknesses_dft, roughnesses_dft, orientations_dft
)
gamma_dft_deg = gamma_dft * 180 / np.pi

# ── Density profile bundles ──────────────────────────────────────────────────
# Free film densities come from the per-slab least-squares fit (slab_results);
# DFT film densities from the UniTensorSLD parameters carried in dft_n.
# Substrate (SiO2, Si) mass densities are read off the MaterialSLD slabs so
# the panel (b) profile steps up through the buried interface. All are
# erf-smoothed through the same thickness/roughness geometry as gamma.

free_rho_means = [slab_results[s]["density"].nominal_value for s in _slab_order_fig]
free_rho_errs  = [slab_results[s]["density"].std_dev for s in _slab_order_fig]

rho_sub_free = _substrate_densities(
    exp_constrained.objectives[0].model.structure,
    len(thicknesses) - len(free_rho_means) - 1,
)

(
    z_rho,
    rho_free,
    rho_free_lo,
    rho_free_hi,
) = density_profile_uncertainty(
    thicknesses,
    roughnesses,
    free_rho_means,
    free_rho_errs,
    rho_sub_free,
    n_samples=4000,
    ci=0.68,
)

dft_rho_means, dft_rho_errs = _density_bundle_from_dft_n(dft_n, _slab_order_dft_fig)

rho_sub_dft = _substrate_densities(
    dft_constrained.objectives[0].model.structure,
    len(thicknesses_dft) - len(dft_rho_means) - 1,
)

(
    z_rho_dft,
    rho_dft,
    rho_dft_lo,
    rho_dft_hi,
) = density_profile_uncertainty(
    thicknesses_dft,
    roughnesses_dft,
    dft_rho_means,
    dft_rho_errs,
    rho_sub_dft,
    n_samples=4000,
    ci=0.68,
)

# Shading alpha channel follows the full mass-density profile (film and
# substrate), normalized to its maximum, so opacity tracks where matter
# actually is: transparent vacuum, intermediate film, darkest Si substrate.
rho_on_z = np.interp(z, z_rho, rho_free)

# ── Region geometry ──────────────────────────────────────────────────────────

region_labels = ["vacuum", "surface", "bulk", "interface", "SiO2", "Si"]
cum_phys      = np.cumsum(np.asarray(thicknesses[1:], dtype=float))
region_edges  = [float(z[0]), 0.0] + cum_phys.tolist()
region_edges[-1] = max(region_edges[-1], float(z[-1]))

# Interior layer boundaries (drop the outermost left and right plot edges).
LAYER_BARRIERS = region_edges[1:-1]

# ── Statistics ───────────────────────────────────────────────────────────────

# Panel (c) displays the likelihood-form criteria (STATS_USE_LOGL = True).
# The chi-square forms differ by the model-independent constant
# sum_i ln(2 pi sigma_i^2), which inflates the absolute scale without
# moving dAIC or dBIC; the sanity block below verifies that agreement on
# every run. The displayed chi2_nu is residual-based in both conventions.
STATS_USE_LOGL = True

stats = {
    "DFT slab":    model_selection_stats(dft_constrained, use_logl=STATS_USE_LOGL),
    "Free tensor": model_selection_stats(exp_constrained, use_logl=STATS_USE_LOGL),
}

# Sanity check: both conventions side by side; the deltas must agree.
_stats_chi2 = {
    "DFT slab":    model_selection_stats(dft_constrained, use_logl=False),
    "Free tensor": model_selection_stats(exp_constrained, use_logl=False),
}
_stats_logl = {
    "DFT slab":    model_selection_stats(dft_constrained, use_logl=True),
    "Free tensor": model_selection_stats(exp_constrained, use_logl=True),
}
for _m in stats:
    _c, _l = _stats_chi2[_m], _stats_logl[_m]
    print(
        f"{_m}: n={_c['npoints']}, k={_c['N']}, chi2_nu={_c['chi2']:.2f} | "
        f"chi2-form AIC={_c['aic']:.1f}, BIC={_c['bic']:.1f} | "
        f"logl-form AIC={_l['aic']:.1f}, BIC={_l['bic']:.1f}"
    )
print(
    "dAIC agreement (chi2-form vs logl-form): "
    f"{_stats_chi2['Free tensor']['aic'] - _stats_chi2['DFT slab']['aic']:.2f} vs "
    f"{_stats_logl['Free tensor']['aic'] - _stats_logl['DFT slab']['aic']:.2f}"
)

# Akaike weights and Bayes factor, evaluated in log space. The pairwise
# ratio exp(-d/2) over- or underflows past |d| ~ 1400, so everything is
# carried as ln and reported as log10. ln BF(DFT/Free) = -dBIC/2 is the
# BIC (Schwarz) estimate of the Bayes factor under equal model priors;
# the analogous AIC quantity is the relative likelihood / evidence ratio.
_LN10 = float(np.log(10.0))
for _key, _label, _ratio_name in (
    ("aic", "Akaike", "evidence ratio"),
    ("bic", "Bayes (BIC)", "Bayes factor"),
):
    _ln_w, _ = log_model_weights(stats, key=_key)
    _ln_ratio = -0.5 * (stats["DFT slab"][_key] - stats["Free tensor"][_key])
    print(
        f"{_label}: d{_key.upper()}(DFT - Free) = "
        f"{stats['DFT slab'][_key] - stats['Free tensor'][_key]:+.1f} | "
        f"log10 {_ratio_name}(DFT/Free) = {_ln_ratio / _LN10:+.2f} | "
        f"P(DFT) = {np.exp(_ln_w['DFT slab']):.6f}, "
        f"log10 P(DFT) = {_ln_w['DFT slab'] / _LN10:.4g}, "
        f"log10 P(Free) = {_ln_w['Free tensor'] / _LN10:.4g}"
    )

# Experimental correction (non-structural) parameter counts for panel (c).
#   Both models: 21 s-pol + 21 p-pol scale factors and
#                21 s-pol + 21 p-pol theta offsets.
#   DFT additionally carries one global energy offset.
N_CORR = {
    "DFT slab":    21 * 4 + 1,
    "Free tensor": 21 * 4,
}

# ── Figure factory ───────────────────────────────────────────────────────────

def make_orientation_figure(
    *,
    show_free_errors,
    show_dft_errors,
    show_shading,
    show_barriers,
    show_cbars = False,
    savename,
):
    """Render one variant of the three-panel orientation figure."""

    fig = plt.figure(figsize=(FIG_W, FIG_H))

    gs_profiles = gridspec.GridSpec(
        2, 1, figure=fig,
        height_ratios=[1.10, 0.90],
        hspace=0.1, top=0.90, bottom=0.42,
    )
    gs_stat = gridspec.GridSpec(
        1, 1, figure=fig, top=0.30, bottom=0.08,
    )

    ax_gamma = fig.add_subplot(gs_profiles[0])
    ax_rho   = fig.add_subplot(gs_profiles[1], sharex=ax_gamma)
    ax_stat  = fig.add_subplot(gs_stat[0])

    def _place_panel_label(ax, letter, *, y=PANEL_LBL_Y):
        ax.yaxis.set_label_coords(YLABEL_X, 0.5)
        ax.text(
            YLABEL_X, y, letter,
            transform=ax.transAxes,
            ha="right", va="bottom",
            fontsize=10,
        )

    def _draw_barriers(ax):
        # White-background variants (v3, v5) only; the shaded variants get
        # their boundaries from shade_layers_p2_density by default.
        for xi in LAYER_BARRIERS:
            ax.axvline(xi, color="grey", ls="--", zorder=5)

    # ── Panel (a): γ vs depth ────────────────────────────────────────────────

    ax_gamma.set_ylabel(r"$\gamma$ (deg)")
    ax_gamma.set_xlim(region_edges[0], z.max())
    ax_gamma.set_ylim(50, 75)

    if show_shading:
        shade_layers_p2_density(
            ax_gamma, z, p2, rho_on_z, region_edges, region_labels, LAYER_CFG
        )
    if show_barriers:
        _draw_barriers(ax_gamma)

    if show_free_errors:
        ax_gamma.fill_between(
            z, gamma_deg_lo, gamma_deg_hi,
            color=C_FREE, alpha=0.30, linewidth=0, zorder=2,
        )
    if show_dft_errors:
        ax_gamma.fill_between(
            z_dft_u, gamma_dft_deg_lo, gamma_dft_deg_hi,
            color=C_DFT, alpha=0.30, linewidth=0, zorder=2,
        )

    ax_gamma.plot(
        z, gamma_deg, color=C_FREE, ls="--", lw=2,
        zorder=4, dashes=dashes,
    )
    ax_gamma.plot(
        z_dft, gamma_dft_deg, color=C_DFT, ls="-", lw=2,
        zorder=3,
    )
    ax_gamma.axhline(
        MAGIC_ANGLE_DEG,
        color=plt.rcParams["axes.edgecolor"],
        lw=plt.rcParams["axes.linewidth"],
    )
    ax_gamma.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_gamma.yaxis.set_minor_locator(ticker.MultipleLocator(2.5))
    ax_gamma.tick_params(axis="y", pad=YTICK_PAD)
    ax_gamma.tick_params(labelbottom=False)

    for lbl in ax_gamma.get_yticklabels():
        lbl.set_horizontalalignment("right")

    # # P2 twin axis. P2(gamma) is monotonic over the displayed range, so the
    # # secondary axis restores the order parameter lost when panel (b) became
    # # the density profile. Mirrored right ticks are disabled to make room.
    # def _g2p(g):
    #     return 1.5 * np.cos(np.radians(g)) ** 2 - 0.5

    # def _p2g(p):
    #     return np.degrees(
    #         np.arccos(np.sqrt(np.clip((2 * np.asarray(p) + 1) / 3, 0, 1)))
    #     )

    # secax = ax_gamma.secondary_yaxis("right", functions=(_g2p, _p2g))
    # secax.set_ylabel(r"$P_2$")
    # secax.yaxis.set_major_locator(ticker.FixedLocator([0.0, -0.125, -0.25]))
    # secax.yaxis.set_major_formatter(
    #     ticker.FixedFormatter(["0", r"$-1/8$", r"$-1/4$"])
    # )
    # secax.yaxis.set_minor_locator(ticker.NullLocator())
    # secax.tick_params(axis="y", pad=YTICK_PAD)
    # ax_gamma.tick_params(axis="y", which="both", right=False)

    # Magic-angle label in the vacuum corner.
    ma_tf = mpl.transforms.blended_transform_factory(
        ax_gamma.transAxes, ax_gamma.transData
    )
    ax_gamma.text(
        0.02, MAGIC_ANGLE_DEG - 0.8, r"54.7$^\circ$",
        transform=ma_tf, ha="left", va="top",
        fontsize=7, color="k",
        path_effects=STROKE,
    )

    # Substrate label inside the gray band.
    ax_gamma.text(
        (region_edges[4] + region_edges[-1]) / 2, 62.5,
        r"SiO$_2$/Si", rotation=90,
        ha="center", va="center",
        fontsize=6.5, color="k",
        path_effects=STROKE,
    )

    # Proxy handles. The on-plot free trace uses sparse (3, 6) dashes so the
    # DFT trace shows through, but at handlelength ~2 that pattern collapses
    # to a single segment and reads as solid. The legend therefore carries
    # its own dense dash pattern that is unambiguous at handle scale.
    legend_handles = [
        Line2D(
            [], [], color=C_FREE, lw=2,
            ls=(0, (2.8, 1.6)), label="Free slab",
        ),
        Line2D(
            [], [], color=C_DFT, lw=2,
            ls="-", label="DFT slab",
        ),
    ]
    ax_gamma.legend(
        handles=legend_handles,
        loc="lower center", ncol=2,
        handlelength=2.8, handleheight=0.8,
        columnspacing=0.8, handletextpad=0.4,
        borderpad=0.0, frameon=False,
    )

    anno_tf = mpl.transforms.blended_transform_factory(
        ax_gamma.transData, ax_gamma.transAxes
    )
    layer_anno = [
        ("surface",   (region_edges[1] + region_edges[2]) / 2),
        ("bulk",      (region_edges[2] + region_edges[3]) / 2),
        ("interface", (region_edges[3] + region_edges[4]) / 2),
    ]
    for lbl, xc in layer_anno:
        ax_gamma.text(
            xc, 1.03, lbl,
            transform=anno_tf,
            ha="center", va="bottom",
            fontsize=7, color="0.30",
            clip_on=False,
        )

    _place_panel_label(ax_gamma, "(a)", y=1.10)

    # ── Panel (b): ρ vs depth ────────────────────────────────────────────────

    ax_rho.set_ylabel(r"$\rho$ (g cm$^{-3}$)")
    ax_rho.set_xlabel(r"Depth ($\AA$)")
    ax_rho.set_xlim(-10, z.max())
    rho_top = 1.15 * max(float(rho_free.max()), float(rho_dft.max()))
    ax_rho.set_ylim(0, rho_top)

    if show_shading:
        shade_layers_p2_density(
            ax_rho, z, p2, rho_on_z, region_edges, region_labels, LAYER_CFG
        )
    if show_barriers:
        _draw_barriers(ax_rho)

    if show_free_errors:
        ax_rho.fill_between(
            z_rho, rho_free_lo, rho_free_hi,
            color=C_FREE, alpha=0.30, linewidth=0, zorder=2,
        )
    if show_dft_errors:
        ax_rho.fill_between(
            z_rho_dft, rho_dft_lo, rho_dft_hi,
            color=C_DFT, alpha=0.30, linewidth=0, zorder=2,
        )

    ax_rho.plot(
        z_rho, rho_free, color=C_FREE, ls="--", lw=2,
        zorder=4, dashes=dashes,
    )
    ax_rho.plot(
        z_rho_dft, rho_dft, color=C_DFT, ls="-", lw=2, zorder=3,
    )

    ax_rho.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_rho.yaxis.set_minor_locator(ticker.MultipleLocator(0.25))
    ax_rho.tick_params(axis="y", pad=YTICK_PAD)

    for lbl in ax_rho.get_yticklabels():
        lbl.set_horizontalalignment("right")

    # Qualitative encoding key for the background shading, tucked into the
    # empty band below the density plateau. Hue strip reproduces the exact
    # P2 mapping (purple -> gray -> amber); opacity strip runs 0 -> rho_Si.
    if show_cbars:
        strip = np.zeros((1, 256, 4))
        for i, t in enumerate(np.linspace(-1, 1, 256)):
            hue = 0.78 if t < 0 else 0.08
            strip[0, i] = (*colorsys.hls_to_rgb(hue, 0.68, abs(t) * 0.90), 1.0)
        ax_o = ax_rho.inset_axes([0.38, 0.30, 0.26, 0.085])
        ax_o.imshow(strip, aspect="auto", origin="lower", extent=[0, 1, 0, 1])
        ax_o.set_xticks([])
        ax_o.set_yticks([])
        for sp in ax_o.spines.values():
            sp.set_linewidth(0.4)
        ax_o.text(-0.07, 0.5, "face-on", transform=ax_o.transAxes,
                ha="right", va="center", fontsize=6.5, color="k",
                path_effects=STROKE)
        ax_o.text(1.07, 0.5, "edge-on", transform=ax_o.transAxes,
                ha="left", va="center", fontsize=6.5, color="k",
                path_effects=STROKE)

        grad = np.linspace(0, 1, 256).reshape(1, -1)
        ax_d = ax_rho.inset_axes([0.38, 0.10, 0.26, 0.085])
        ax_d.imshow(grad, aspect="auto", cmap="gray_r", vmin=0, vmax=1,
                    extent=[0, 1, 0, 1])
        ax_d.set_xticks([])
        ax_d.set_yticks([])
        for sp in ax_d.spines.values():
            sp.set_linewidth(0.4)
        ax_d.text(-0.07, 0.5, "Vac", transform=ax_d.transAxes,
                ha="right", va="center", fontsize=6.5, color="k",
                path_effects=STROKE)
        ax_d.text(1.07, 0.5, "Si", transform=ax_d.transAxes,
                ha="left", va="center", fontsize=6.5, color="k",
                path_effects=STROKE)

    _place_panel_label(ax_rho, "(b)")

    # ── Panel (c): statistics bar chart ─────────────────────────────────────

    METRICS   = [r"$\chi^2_\nu$", "N", "AIC", "BIC"]
    BAR_W     = 0.28
    GROUP_GAP = 0.82
    INTRA_GAP = 0.04

    models     = list(stats.keys())
    bar_colors = [C_DFT, C_FREE]
    mkeys      = ["chi2", "N", "aic", "bic"]

    gcentres = np.arange(len(METRICS)) * GROUP_GAP
    offsets = np.linspace(
        -((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
        ((len(models) - 1) * (BAR_W + INTRA_GAP) / 2),
        len(models),
    )

    raw_vals = {mk: np.array([stats[m][mk] for m in models]) for mk in mkeys}
    norms    = {mk: v.max() for mk, v in raw_vals.items()}

    for gi, mk in enumerate(mkeys):
        for bi, (model, color) in enumerate(zip(models, bar_colors)):
            raw  = stats[model][mk]
            norm = raw / norms[mk]
            x    = gcentres[gi] + offsets[bi]
            if mk == "N":
                # Experimental correction floor (per-energy scale factors
                # and theta offsets, plus the DFT energy offset), hatched;
                # structural parameters stack solid on top.
                corr = N_CORR[model] / norms[mk]
                ax_stat.bar(
                    x, corr, width=BAR_W, color=color,
                    alpha=0.90, linewidth=0, zorder=2,
                    hatch="/////", edgecolor="white",
                )
                ax_stat.bar(
                    x, norm - corr, bottom=corr, width=BAR_W, color=color,
                    alpha=0.90, linewidth=0, zorder=2,
                )
            else:
                ax_stat.bar(
                    x, norm, width=BAR_W, color=color,
                    alpha=0.90, linewidth=0, zorder=2,
                )
            val = raw - N_CORR[model] if mk == "N" else raw
            fmt = f"{val:.1f}" if mk == "chi2" else f"{val:.0f}"
            ax_stat.text(
                x, norm + 0.025, fmt,
                ha="center", va="bottom", fontsize=8,
            )

    ax_stat.set_ylim(0, 1.4)

    # Bayes factor and Akaike weight as arithmetic values, assembled from
    # log10 so the exponents survive. BF uses dBIC regardless of the
    # STATS_USE_LOGL flag since the deltas agree between conventions.
    def _sci_tex(l10):
        e = int(np.floor(l10))
        m = 10.0 ** (l10 - e)
        return rf"{m:.1f}\times10^{{{e}}}"

    _ln10 = np.log(10.0)
    bf_l10 = -0.5 * (stats["DFT slab"]["bic"] - stats["Free tensor"]["bic"]) / _ln10
    _dA = stats["Free tensor"]["aic"] - stats["DFT slab"]["aic"]
    w_free_l10 = (-0.5 * _dA - np.logaddexp(0.0, -0.5 * _dA)) / _ln10
    ax_stat.text(
        (gcentres[0] + gcentres[-1]) / 2, 1.35,
        rf"$\mathrm{{BF}} = {_sci_tex(bf_l10)}\;\;\;"
        rf"w^{{\mathrm{{AIC}}}}_{{\mathrm{{Free}}}} = {_sci_tex(w_free_l10)}$",
        ha="center", va="top", fontsize=6.5, color="k",
    )

    # Hatch key for the experimental-correction floor of the N bars.
    ax_stat.annotate(
        "exp. corr.",
        xy=(gcentres[1], 0.05),
        xytext=(gcentres[1], 0.08),
        fontsize=6.5, color="k",
        ha="center", va="bottom",
        path_effects=[
            pe.Stroke(linewidth=2.5, foreground="white"),
            pe.Normal(),
        ],
    )

    ax_stat.set_xticks(gcentres)
    ax_stat.set_xticklabels(METRICS)
    ax_stat.set_yticklabels([])
    ax_stat.grid(False)
    ax_stat.axhline(1.0, color="0.65", lw=0.5, ls="--", zorder=1)
    ax_stat.set_ylabel(r"AIC / BIC / $\chi^2$")

    # Down arrow just outside the y spine. Lower is better for all model
    # selection criteria shown. annotation_clip=False keeps it through
    # bbox_inches="tight".
    ax_stat.annotate(
        "",
        xy=(-0.045, 0.05), xytext=(-0.045, 0.92),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(
            arrowstyle="-|>", color="0.30", lw=0.9,
            shrinkA=0, shrinkB=0, mutation_scale=8,
        ),
        annotation_clip=False,
    )
    ax_stat.text(
        -0.068, 0.585, "better",
        transform=ax_stat.transAxes,
        rotation=90, ha="center", va="center",
        fontsize=6.5, color="0.30",
    )

    _place_panel_label(ax_stat, "(c)")

    plt.savefig(
        savename, format="png", dpi=600,
        bbox_inches="tight", transparent=True,
    )
    plt.show()


# v1  nominal traces only, region shading
make_orientation_figure(
    show_free_errors=False,
    show_dft_errors=False,
    show_shading=True,
    show_barriers=False,
    savename="fig3_orientation.png",
)

In [ ]:
"""
Figure 1 (single PRL column): per-energy free tensor fits vs the DFT slab model.

Argument
  The free model fits the optical tensor independently at each probe energy,
  so its raw outputs are the per-energy birefringence of each slab, each with
  a fit uncertainty. The DFT slab model reproduces those spectra from two
  parameters per slab (rotation, density) and threads the free points with a
  propagated uncertainty band. Recovering the quantity you report, gamma and
  density, then takes a second-stage fit from the free model but comes
  directly from the DFT model. The per-energy free estimates scatter widely
  (box-and-whisker) while the DFT value is far tighter, because a one-stage
  joint fit to the reflectivity beats the two-stage compress-then-fit.

Layout (single column, 3.35 in)
  (a) birefringence vs energy, surface     DFT trace + band, free points + err
  (b) birefringence vs energy, bulk
  (c) birefringence vs energy, interface
  (d) gamma forest  |  (e) density forest   box = free per-energy spread,
                                            square = DFT
  (f) MSC bars (carried from fig3)

Styling note
  Font sizes, tick sizes, and line defaults come from the global rcParams
  (set_plotting_defaults). Nothing here overrides them; only data encodings
  (color, marker shape, structural visibility) are set locally.

Prerequisites (fig3 pipeline)
  free_n        per-(slab, energy) tensor fit, with birefringence/_err,
                and xx, zz, ixx, izz (+ _err) for the per-energy inversion
  dft_n         per-slab DFT tensor traces (birefringence) + rotation,
                rotation_err, density (ufloat)
  slab_results  free joint-fit gamma, density ufloats per slab
  ooc           oriented optical constants (n_xx, n_zz, n_ixx, n_izz vs energy)
  dft_constrained, exp_constrained   GlobalObjectives, for the MSC bars
"""

from matplotlib import gridspec
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from scipy.optimize import least_squares

PALETTE = ["C0", "C1", "C2", "C3", "C4", "C5"]
C_DFT = PALETTE[0]
C_FREE = PALETTE[1]

LB, UB = 281.0, 290.0
MAGIC_ANGLE_DEG = np.degrees(np.arcsin(np.sqrt(2 / 3)))
slab_order = ["surface", "bulk", "interface"]
slab_names = ["surf.", "bulk", "int."]
biref_bounds = [(-2.5e-3, 2.5e-3), (-2.5e-3, 2.5e-3), (-2.5e-3, 2.5e-3)]

# All three birefringence panels share one decade; factor it into the common
# y-label rather than printing a per-panel offset.
BIREF_SCALE = 1e-3

FIG_W, FIG_H = 3.35, 5.8

# ── Per-energy inversion: free tensor -> apparent (gamma, density) ───────────

def _apparent_gamma_density(row, ooc_df, gamma0, density0):
    """
    Fit (gamma, density) to a single energy's free tensor components.

    Mirrors fit_orientation_from_free but for one energy: four observations
    (ixx, izz, xx, zz) against the oriented optical constants at that energy.
    Returns (gamma_rad, density) or None if the inversion fails (near the
    isotropic point birefringence vanishes and the problem is ill-posed,
    which is itself the source of the per-energy spread).
    """
    e = float(row["energy"])
    ixx = float(np.interp(e, ooc_df["energy"], ooc_df["n_ixx"]))
    izz = float(np.interp(e, ooc_df["energy"], ooc_df["n_izz"]))
    xx0 = float(np.interp(e, ooc_df["energy"], ooc_df["n_xx"]))
    zz0 = float(np.interp(e, ooc_df["energy"], ooc_df["n_zz"]))

    obs = np.array([row["ixx"], row["izz"], row["xx"], row["zz"]], dtype=float)
    err = np.array([
        max(abs(row.get("ixx_err", 0.0)), 1e-12),
        max(abs(row.get("izz_err", 0.0)), 1e-12),
        max(abs(row.get("xx_err", 0.0)), 1e-12),
        max(abs(row.get("zz_err", 0.0)), 1e-12),
    ])

    def resid(p):
        g, d = p
        cg, sg = np.cos(g), np.sin(g)
        m_ixx = d * (ixx * (cg + 1) + izz * sg) / 2
        m_xx = d * (xx0 * (cg + 1) + zz0 * sg) / 2
        m_izz = d * (ixx * sg + izz * cg)
        m_zz = d * (xx0 * sg + zz0 * cg)
        return (obs - np.array([m_ixx, m_izz, m_xx, m_zz])) / err

    try:
        res = least_squares(
            resid, x0=[gamma0, density0], method="trf",
            bounds=([0.0, 1], [np.pi / 2, 3]),
            max_nfev=200,
        )
    except Exception:
        return None
    if not res.success:
        return None
    return float(res.x[0]), float(res.x[1])


def per_energy_cloud(slab, ooc_df, gamma0, density0):
    """Apparent gamma (deg) and density across this slab's probe energies."""
    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    gammas, dens = [], []
    for _, row in f.iterrows():
        out = _apparent_gamma_density(row, ooc_df, gamma0, density0)
        if out is None:
            continue
        g, d = out
        gammas.append(np.degrees(g))
        dens.append(d)
    return np.array(gammas), np.array(dens)

# ── Forest scalars ───────────────────────────────────────────────────────────

def _pair(value):
    n = value.n if hasattr(value, "n") else float(value)
    s = value.s if hasattr(value, "s") else 0.0
    return float(n), float(s)


def _dft_scalar(slab, col):
    return _pair(dft_n[dft_n["slab"] == slab].iloc[0][col])


g_dft = {s: np.degrees(_dft_scalar(s, "rotation")[0]) for s in slab_order}
g_dft_e = {s: np.degrees(_dft_scalar(s, "rotation")[1]) for s in slab_order}
r_dft = {s: _dft_scalar(s, "density")[0] for s in slab_order}
r_dft_e = {s: _dft_scalar(s, "density")[1] for s in slab_order}

clouds_g, clouds_r = {}, {}
for s in slab_order:
    g0 = slab_results[s]["gamma"].nominal_value
    d0 = slab_results[s]["density"].nominal_value
    clouds_g[s], clouds_r[s] = per_energy_cloud(s, ooc, g0, d0)
    print(
        f"{s}: per-energy gamma n={clouds_g[s].size}, "
        f"IQR={np.subtract(*np.percentile(clouds_g[s], [75, 25])):.2f} deg, "
        f"DFT err={g_dft_e[s]:.2f} deg"
    )

# ── Statistics (same machinery as fig3) ──────────────────────────────────────

def model_selection_stats(objective, *, use_logl=True):
    resid = np.asarray(objective.residuals(), dtype=float)
    n = int(resid.size)
    k = len(objective.varying_parameters())
    chi2 = float(np.sum(resid**2))
    nu = max(n - k, 1)
    if use_logl:
        logl = float(objective.logl())
        aic = 2.0 * k - 2.0 * logl
        bic = k * np.log(n) - 2.0 * logl
    else:
        aic = chi2 + 2.0 * k
        bic = chi2 + k * np.log(n)
    return {"chi2": chi2 / nu, "aic": aic, "bic": bic, "N": k, "npoints": n}


stats = {
    "DFT slab": model_selection_stats(dft_constrained),
    "Free tensor": model_selection_stats(exp_constrained),
}
N_CORR = {"DFT slab": 21 * 4 + 1, "Free tensor": 21 * 4}

# ── Figure ───────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(FIG_W, FIG_H))
outer = gridspec.GridSpec(
    3, 1, figure=fig, height_ratios=[3.1, 1.25, 1.0],
    hspace=0.34,
)
gs_spec = outer[0].subgridspec(3, 1, hspace=0.08)
gs_for = outer[1].subgridspec(1, 2, wspace=0.08)

axS = [fig.add_subplot(gs_spec[i]) for i in range(3)]
for a in axS[:-1]:
    a.sharex(axS[-1])
ax_g = fig.add_subplot(gs_for[0])
ax_r = fig.add_subplot(gs_for[1], sharey=ax_g)
ax_m = fig.add_subplot(outer[2])

emk = dict(mfc="white", mec=C_FREE, mew=1.0, ls="none", ecolor=C_FREE,
           elinewidth=0.8, capsize=1.5, ms=4)

# Ticks display value / BIREF_SCALE; the decade lives in the shared label.
_biref_fmt = FuncFormatter(lambda v, _: f"{v / BIREF_SCALE:.0f}")


def _spectral(ax, slab, i, last=False):
    d = dft_n[dft_n["slab"] == slab]
    d = d[(d["energy"] >= LB) & (d["energy"] <= UB)].sort_values("energy")
    if "birefringence_err" in d.columns:
        ax.fill_between(d["energy"],
                        d["birefringence"] - d["birefringence_err"],
                        d["birefringence"] + d["birefringence_err"],
                        color=C_DFT, alpha=0.25, lw=0, zorder=2)
    ax.plot(d["energy"], d["birefringence"], "-", color=C_DFT, zorder=3)

    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    ax.errorbar(f["energy"], f["birefringence"], yerr=f["birefringence_err"],
                marker="o", zorder=4, **emk)

    ax.axhline(0, color="0.6", lw=0.5, zorder=1)
    ax.set_xlim(LB, UB)
    ax.set_xticks([282, 284, 286, 288])

    ax.set_yticks([-2e-3, -1e-3, 0, 1e-3, 2e-3])
    ax.set_ylim(biref_bounds[i])
    ax.yaxis.set_major_formatter(_biref_fmt)   # no per-panel offset text
    ax.text(0.97, 0.90, slab, transform=ax.transAxes, ha="right", va="top",
            color="0.30")
    if not last:
        ax.tick_params(labelbottom=False)


for i, s in enumerate(slab_order):
    _spectral(axS[i], s, i, last=(i == 2))
axS[2].set_xlabel(r"Energy (eV)", labelpad=-1)

# Single shared y-label across the three spectra, scale folded in. Centered on
# the middle panel, which sits at the vertical center of the equal-height stack.
axS[1].set_ylabel(r"$\Delta\delta$ ($\times 10^{-3}$)")
axS[1].yaxis.set_label_coords(-0.1, 0.5)

# ── Forests: free per-energy box-whisker + DFT marker, shared y ──────────────

ypos = {s: i for i, s in enumerate(slab_order)}


def _forest(ax, clouds, dft_val, dft_err, xlabel, leftmost):
    data = [clouds[s] for s in slab_order]
    positions = [ypos[s] for s in slab_order]
    ax.boxplot(
        data, positions=positions, vert=False, widths=0.55, whis=.6827,
        showfliers=False, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker=".", mfc=C_FREE, mec=C_FREE, alpha=0.5),
        medianprops=dict(color=C_FREE),
        whiskerprops=dict(color=C_FREE),
        capprops=dict(color=C_FREE),
        boxprops=dict(facecolor=C_FREE, alpha=0.20, edgecolor=C_FREE),
    )
    for s in slab_order:
        y = ypos[s]
        ax.errorbar(dft_val[s], y, xerr=dft_err[s], marker="s",
                    color=C_DFT, ecolor=C_DFT, capsize=2, zorder=6, ms=4)
    ax.set_yticks(list(ypos.values()))
    ax.set_ylim(-1, len(slab_order))
    ax.invert_yaxis()
    ax.set_xlabel(xlabel, labelpad=-2)
    if leftmost:
        ax.set_yticklabels(slab_names, rotation=45)
    else:
        ax.tick_params(labelleft=False)


_forest(ax_g, clouds_g, g_dft, g_dft_e, r"$\gamma$ (deg)", leftmost=True)
ax_g.axvline(MAGIC_ANGLE_DEG, color="0.35", lw=0.6, zorder=0)
_forest(ax_r, clouds_r, r_dft, r_dft_e, r"$\rho$ (g cm$^{-3}$)", leftmost=False)
ax_r.axvline(1.61, color="0.35", lw=0.6, zorder=0)

leg_handles = [
    Line2D([], [], color=C_DFT, marker="s", label="DFT Slab"),
    Line2D([], [], color=C_FREE, marker="s", mfc=(0.17, 0.63, 0.17, 0.20),
           ls="none", label="Free Slab"),
]

# ── MSC bars (carried from fig3) ─────────────────────────────────────────────

METRICS = [r"$\chi^2_\nu$", "N", "AIC", "BIC"]
BAR_W, GROUP_GAP, INTRA_GAP = 0.30, 0.85, 0.04
models = list(stats)
bar_colors = [C_DFT, C_FREE]
mkeys = ["chi2", "N", "aic", "bic"]
gc = np.arange(4) * GROUP_GAP
offs = np.linspace(-(BAR_W + INTRA_GAP) / 2, (BAR_W + INTRA_GAP) / 2, 2)
norms = {mk: max(stats[m][mk] for m in models) for mk in mkeys}
for gi, mk in enumerate(mkeys):
    for bi, (m, col) in enumerate(zip(models, bar_colors)):
        raw = stats[m][mk]
        norm = raw / norms[mk]
        x = gc[gi] + offs[bi]
        if mk == "N":
            corr = N_CORR[m] / norms[mk]
            ax_m.bar(x, corr, BAR_W, color=col, alpha=0.9, lw=0,
                     hatch="/////", edgecolor="white")
            ax_m.bar(x, norm - corr, BAR_W, bottom=corr, color=col, alpha=0.9, lw=0)
        else:
            ax_m.bar(x, norm, BAR_W, color=col, alpha=0.9, lw=0)
        ax_m.text(x, norm + 0.02, f"{raw:.1f}" if mk == "chi2" else f"{raw:.0f}",
                  ha="center", va="bottom")
ax_m.set_ylim(0, 1.5)
ax_m.set_xticks(gc)
ax_m.set_xticklabels(METRICS)
ax_m.set_yticklabels([])
ax_m.axhline(1.0, color="0.65", lw=0.5, ls="--")
# add space between MSC label and the plot for the arrow
ax_m.set_ylabel("MSC", labelpad=17)

# Two-entry legend below the MSC panel, clear of the forest x-axis labels in
# the gap above and of the bar value labels inside.
ax_m.legend(handles=leg_handles, loc="upper center", bbox_to_anchor=(0.5, -0.30),
            ncol=2, frameon=False)

# "lower is better" arrow in the left margin.
ax_m.annotate("", xy=(-0.02, 0.12), xytext=(-0.02, 0.88),
              xycoords="axes fraction", textcoords="axes fraction",
              arrowprops=dict(arrowstyle="-|>", color="0.3", lw=0.9,
                              shrinkA=0, shrinkB=0, mutation_scale=8),
              annotation_clip=False)
ax_m.text(-0.06, 0.5, "better", transform=ax_m.transAxes, rotation=90,
          ha="center", va="center", color="0.3")

# ── Panel letters ────────────────────────────────────────────────────────────

for ax, lab in zip([axS[0], ax_g, ax_m],
                   ["(a)",  "(d)", "(f)"]):
    if ax == ax_g:
        ax.text(-.36, 1, lab, transform=ax.transAxes, va="top", ha="left")
    else:
        ax.text(-.17, 1, lab, transform=ax.transAxes, va="top", ha="left")

fig.savefig("fig3_orientation.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig.savefig("fig3_orientation.png", dpi=600, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
"""
Figure 3 (single PRL column): per-energy free tensor fits vs the DFT slab model.

Argument
  The free model fits the optical tensor independently at each probe energy,
  so its raw outputs are the per-energy birefringence and dichroism of each
  slab, each with a fit uncertainty. The DFT slab model reproduces those
  spectra from two parameters per slab (rotation, density) and threads the
  free points with a propagated uncertainty band. The apparent per-energy
  (gamma, density) estimates scatter widely (inset box-and-whisker) while
  the DFT value is far tighter, because a one-stage joint fit to the
  reflectivity beats the two-stage compress-then-fit.

Layout (single column, 3.35 in)
  (a) three stacked spectra (surface / bulk / interface): DFT birefringence
      (solid) and dichroism (dashed) traces + free per-energy points
      (circles / triangles), with a per-slab DFT gamma readout
  (b) MSC bars

make_si_forest_figure() exports the stacked SI forest:
  (a) orientation  (b) density
  x-limits are the fit bounds of the per-energy inversion.

Styling note
  Font sizes, tick sizes, and line defaults come from the global rcParams
  (set_plotting_defaults). Only the insets override sizes, since 5 pt inset
  ticks have no rc slot.

Prerequisites (fig3 pipeline)
  free_n        per-(slab, energy) tensor fit, with birefringence/_err,
                dichroism/_err, and xx, zz, ixx, izz (+ _err)
  dft_n         per-slab DFT tensor traces (birefringence, dichroism)
                + rotation, rotation_err, density (ufloat)
  slab_results  free joint-fit gamma, density ufloats per slab
  ooc           oriented optical constants (n_xx, n_zz, n_ixx, n_izz vs energy)
  dft_constrained, exp_constrained   GlobalObjectives, for the MSC bars
"""

from matplotlib import gridspec, ticker
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from scipy.optimize import least_squares
from uncertainties import ufloat

PALETTE = ["C0", "C1", "C2", "C3", "C4", "C5"]
C_DFT = PALETTE[0]
C_FREE = PALETTE[1]

LB, UB = 281.0, 290.0
MAGIC_ANGLE_DEG = np.degrees(np.arcsin(np.sqrt(2 / 3)))
RHO_NOMINAL = 1.61
slab_order = ["surface", "bulk", "interface"]
slab_names = ["surf.", "bulk", "int."]

# Panel y-bounds are computed per slab from both quantities (traces and free
# points including error bars), symmetric about zero, padded by Y_PAD, and
# clipped at Y_CAP so a single huge pre-edge error bar cannot flatten the
# traces. Bars beyond Y_CAP clip visibly, which is honest.
Y_PAD = 1.10
Y_CAP = 4.5e-3

# Fit bounds for the per-energy (gamma, density) inversion. The SI forest
# axes use exactly these limits, so the displayed window IS the allowed
# parameter space and clipping against it is visible.
GAMMA_FIT_BOUNDS = (0.0, np.pi / 2)          # rad
RHO_FIT_BOUNDS = (1.0, 3.0)                  # g cm^-3

# All three spectra panels share one decade; factor it into the common
# y-label rather than printing a per-panel offset.
BIREF_SCALE = 1e-3

FIG_W, FIG_H = 3.35, 5.8

# ── Per-energy inversion: free tensor -> apparent (gamma, density) ───────────

def _apparent_gamma_density(row, ooc_df, gamma0, density0):
    """
    Fit (gamma, density) to a single energy's free tensor components.

    Mirrors fit_orientation_from_free but for one energy: four observations
    (ixx, izz, xx, zz) against the oriented optical constants at that energy.
    Returns (gamma_rad, density) or None if the inversion fails (near the
    isotropic point birefringence vanishes and the problem is ill-posed,
    which is itself the source of the per-energy spread).
    """
    e = float(row["energy"])
    ixx = float(np.interp(e, ooc_df["energy"], ooc_df["n_ixx"]))
    izz = float(np.interp(e, ooc_df["energy"], ooc_df["n_izz"]))
    xx0 = float(np.interp(e, ooc_df["energy"], ooc_df["n_xx"]))
    zz0 = float(np.interp(e, ooc_df["energy"], ooc_df["n_zz"]))

    obs = np.array([row["ixx"], row["izz"], row["xx"], row["zz"]], dtype=float)
    err = np.array([
        max(abs(row.get("ixx_err", 0.0)), 1e-12),
        max(abs(row.get("izz_err", 0.0)), 1e-12),
        max(abs(row.get("xx_err", 0.0)), 1e-12),
        max(abs(row.get("zz_err", 0.0)), 1e-12),
    ])

    def resid(p):
        g, d = p
        cg, sg = np.cos(g), np.sin(g)
        m_ixx = d * (ixx * (cg + 1) + izz * sg) / 2
        m_xx = d * (xx0 * (cg + 1) + zz0 * sg) / 2
        m_izz = d * (ixx * sg + izz * cg)
        m_zz = d * (xx0 * sg + zz0 * cg)
        return (obs - np.array([m_ixx, m_izz, m_xx, m_zz])) / err

    try:
        res = least_squares(
            resid, x0=[gamma0, density0], method="trf",
            bounds=([GAMMA_FIT_BOUNDS[0], RHO_FIT_BOUNDS[0]],
                    [GAMMA_FIT_BOUNDS[1], RHO_FIT_BOUNDS[1]]),
            max_nfev=200,
        )
    except Exception:
        return None
    if not res.success:
        return None
    return float(res.x[0]), float(res.x[1])


def per_energy_cloud(slab, ooc_df, gamma0, density0):
    """Apparent gamma (deg) and density across this slab's probe energies."""
    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    gammas, dens = [], []
    for _, row in f.iterrows():
        out = _apparent_gamma_density(row, ooc_df, gamma0, density0)
        if out is None:
            continue
        g, d = out
        gammas.append(np.degrees(g))
        dens.append(d)
    return np.array(gammas), np.array(dens)

# ── Forest scalars ───────────────────────────────────────────────────────────

def _pair(value):
    n = value.n if hasattr(value, "n") else float(value)
    s = value.s if hasattr(value, "s") else 0.0
    return float(n), float(s)


def _dft_scalar(slab, col):
    return _pair(dft_n[dft_n["slab"] == slab].iloc[0][col])


g_dft = {s: np.degrees(_dft_scalar(s, "rotation")[0]) for s in slab_order}
g_dft_e = {s: np.degrees(_dft_scalar(s, "rotation")[1]) for s in slab_order}
r_dft = {s: _dft_scalar(s, "density")[0] for s in slab_order}
r_dft_e = {s: _dft_scalar(s, "density")[1] for s in slab_order}

g_free_pt = {s: _pair(slab_results[s]["gamma"]) for s in slab_order}
g_free_pt = {s: (np.degrees(v), np.degrees(e)) for s, (v, e) in g_free_pt.items()}
r_free_pt = {s: _pair(slab_results[s]["density"]) for s in slab_order}

clouds_g, clouds_r = {}, {}
for s in slab_order:
    g0 = slab_results[s]["gamma"].nominal_value
    d0 = slab_results[s]["density"].nominal_value
    clouds_g[s], clouds_r[s] = per_energy_cloud(s, ooc, g0, d0)
    print(
        f"{s}: per-energy gamma n={clouds_g[s].size}, "
        f"IQR={np.subtract(*np.percentile(clouds_g[s], [75, 25])):.2f} deg, "
        f"DFT err={g_dft_e[s]:.2f} deg"
    )

# ── Statistics (same machinery as fig3) ──────────────────────────────────────

def model_selection_stats(objective, *, use_logl=True):
    resid = np.asarray(objective.residuals(), dtype=float)
    n = int(resid.size)
    k = len(objective.varying_parameters())
    chi2 = float(np.sum(resid**2))
    nu = max(n - k, 1)
    if use_logl:
        logl = float(objective.logl())
        aic = 2.0 * k - 2.0 * logl
        bic = k * np.log(n) - 2.0 * logl
    else:
        aic = chi2 + 2.0 * k
        bic = chi2 + k * np.log(n)
    return {"chi2": chi2 / nu, "aic": aic, "bic": bic, "N": k, "npoints": n}


stats = {
    "DFT slab": model_selection_stats(dft_constrained),
    "Free tensor": model_selection_stats(exp_constrained),
}
N_CORR = {"DFT slab": 21 * 4 + 1, "Free tensor": 21 * 4}

# ── Per-slab spectral y-bounds ───────────────────────────────────────────────

def _panel_bounds(slab, pad=Y_PAD, cap=Y_CAP):
    """Symmetric y-bounds covering biref and dichroism, traces and points."""
    d = dft_n[dft_n["slab"] == slab]
    d = d[(d["energy"] >= LB) & (d["energy"] <= UB)]
    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    vals = [
        d["birefringence"].to_numpy(dtype=float),
        d["dichroism"].to_numpy(dtype=float),
        (f["birefringence"].abs() + f["birefringence_err"].abs()).to_numpy(dtype=float),
        (f["dichroism"].abs() + f["dichroism_err"].abs()).to_numpy(dtype=float),
    ]
    if "birefringence_err" in d.columns:
        vals.append(
            (d["birefringence"].abs() + d["birefringence_err"].abs()).to_numpy(dtype=float)
        )
    if "dichroism_err" in d.columns:
        vals.append(
            (d["dichroism"].abs() + d["dichroism_err"].abs()).to_numpy(dtype=float)
        )
    m = pad * max(float(np.max(np.abs(v))) for v in vals if v.size)
    m = min(m, cap)
    return (-m, m)


spec_bounds = {s: _panel_bounds(s) for s in slab_order}

# ── Shared bits ──────────────────────────────────────────────────────────────

emk = dict(mfc="white", mec=C_FREE, mew=1.0, ls="none", ecolor=C_FREE,
           elinewidth=0.8, capsize=1.5, ms=3.5)

# Ticks display value / BIREF_SCALE; the decade lives in the shared label.
_biref_fmt = FuncFormatter(lambda v, _: f"{v / BIREF_SCALE:.0f}")

DASH_DIC = (0, (2.2, 1.4))

leg_handles = [
    Line2D([], [], color=C_DFT, marker="s", label="DFT Slab"),
    Line2D([], [], color=C_FREE, marker="s", mfc=(0.17, 0.63, 0.17, 0.20),
           ls="none", label="Free Slab"),
]

ypos = {s: i for i, s in enumerate(slab_order)}


def _free_box(ax, data, position, width=0.55):
    ax.boxplot(
        [data], positions=[position], vert=False, widths=width, whis=0.6827,
        showfliers=False, patch_artist=True, manage_ticks=False,
        medianprops=dict(color=C_FREE),
        whiskerprops=dict(color=C_FREE),
        capprops=dict(color=C_FREE),
        boxprops=dict(facecolor=C_FREE, alpha=0.20, edgecolor=C_FREE),
    )

# ── Figure ───────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(FIG_W, FIG_H))
outer = gridspec.GridSpec(
    2, 1, figure=fig, height_ratios=[4.3, 1.0], hspace=0.26,
)
gs_spec = outer[0].subgridspec(3, 1, hspace=0.08)

axS = [fig.add_subplot(gs_spec[i]) for i in range(3)]
for a in axS[:-1]:
    a.sharex(axS[-1])
ax_m = fig.add_subplot(outer[1])


def _spectral(ax, slab, i, last=False):
    d = dft_n[dft_n["slab"] == slab]
    d = d[(d["energy"] >= LB) & (d["energy"] <= UB)].sort_values("energy")
    if "birefringence_err" in d.columns:
        ax.fill_between(d["energy"],
                        d["birefringence"] - d["birefringence_err"],
                        d["birefringence"] + d["birefringence_err"],
                        color=C_DFT, alpha=0.25, lw=0, zorder=2)
    ax.plot(d["energy"], d["birefringence"], "-", color=C_DFT, zorder=3)
    if "dichroism_err" in d.columns:
        ax.fill_between(d["energy"],
                        d["dichroism"] - d["dichroism_err"],
                        d["dichroism"] + d["dichroism_err"],
                        color=C_DFT, alpha=0.18, lw=0, zorder=2)
    ax.plot(d["energy"], d["dichroism"], ls=DASH_DIC, color=C_DFT, zorder=3)

    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    ax.errorbar(f["energy"], f["birefringence"], yerr=f["birefringence_err"],
                marker="o", zorder=4, **emk)
    ax.errorbar(f["energy"], f["dichroism"], yerr=f["dichroism_err"],
                marker="^", zorder=4, **emk)

    ax.axhline(0, color="0.6", lw=0.5, zorder=1)
    ax.set_xlim(LB, UB)
    ax.set_xticks([282, 284, 286, 288])

    ax.set_ylim(spec_bounds[slab])
    # Tick step adapts to the panel window: 1e-3 up to ~3e-3 half-range,
    # 2e-3 beyond, so wide panels stay labeled without crowding.
    step = 1e-3 if spec_bounds[slab][1] <= 3.2e-3 else 2e-3
    ax.yaxis.set_major_locator(ticker.MultipleLocator(step))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(step / 2))
    ax.yaxis.set_major_formatter(_biref_fmt)   # no per-panel offset text
    ax.text(0.03, 0.93, slab, transform=ax.transAxes, ha="left", va="top",
            color="0.30")
    # DFT orientation readout for this slab.
    gamma = ufloat(g_dft[slab], g_dft_e[slab]*4)
    ax.text(
        0.97, 0.93,
        rf"$\gamma = {gamma:.1uS}^\circ$",
        transform=ax.transAxes, ha="right", va="top", color=C_DFT,
    )
    if not last:
        ax.tick_params(labelbottom=False)


for i, s in enumerate(slab_order):
    _spectral(axS[i], s, i, last=(i == 2))
axS[2].set_xlabel(r"Energy (eV)", labelpad=-1)

# Single shared y-label across the three spectra, scale folded in. Centered on
# the middle panel, which sits at the vertical center of the equal-height stack.
axS[1].set_ylabel(r"$\Delta\delta$, $\Delta\beta$ ($\times 10^{-3}$)")
axS[1].yaxis.set_label_coords(-0.1, 0.5)

# Quantity legend above the stack; model colors are keyed under the MSC panel.
q_handles = [
    Line2D([], [], color="0.2", ls="-", marker="o", mfc="white", ms=3.5,
           label=r"$\Delta\delta$ (biref.)"),
    Line2D([], [], color="0.2", ls=DASH_DIC, marker="^", mfc="white", ms=3.5,
           label=r"$\Delta\beta$ (dichro.)"),
]
axS[0].legend(handles=q_handles, loc="lower center", bbox_to_anchor=(0.5, 1.0),
              ncol=2, frameon=False, handlelength=1.8,
              columnspacing=1.0, handletextpad=0.5, borderpad=0.1)

# ── MSC bars (carried from fig3) ─────────────────────────────────────────────

METRICS = [r"$\chi^2_\nu$", "N", "AIC", "BIC"]
BAR_W, GROUP_GAP, INTRA_GAP = 0.30, 0.85, 0.04
models = list(stats)
bar_colors = [C_DFT, C_FREE]
mkeys = ["chi2", "N", "aic", "bic"]
gc = np.arange(4) * GROUP_GAP
offs = np.linspace(-(BAR_W + INTRA_GAP) / 2, (BAR_W + INTRA_GAP) / 2, 2)
norms = {mk: max(stats[m][mk] for m in models) for mk in mkeys}
for gi, mk in enumerate(mkeys):
    for bi, (m, col) in enumerate(zip(models, bar_colors)):
        raw = stats[m][mk]
        norm = raw / norms[mk]
        x = gc[gi] + offs[bi]
        if mk == "N":
            corr = N_CORR[m] / norms[mk]
            ax_m.bar(x, corr, BAR_W, color=col, alpha=0.9, lw=0,
                     hatch="/////", edgecolor="white")
            ax_m.bar(x, norm - corr, BAR_W, bottom=corr, color=col, alpha=0.9, lw=0)
        else:
            ax_m.bar(x, norm, BAR_W, color=col, alpha=0.9, lw=0)
        ax_m.text(x, norm + 0.02, f"{raw:.1f}" if mk == "chi2" else f"{raw:.0f}",
                  ha="center", va="bottom")
ax_m.set_ylim(0, 1.5)
ax_m.set_xticks(gc)
ax_m.set_xticklabels(METRICS)
ax_m.set_yticklabels([])
ax_m.axhline(1.0, color="0.65", lw=0.5, ls="--")
# add space between MSC label and the plot for the arrow
ax_m.set_ylabel("MSC", labelpad=17)

# Two-entry model legend below the MSC panel.
ax_m.legend(handles=leg_handles, loc="upper center", bbox_to_anchor=(0.5, -0.30),
            ncol=2, frameon=False)

# "lower is better" arrow in the left margin.
ax_m.annotate("", xy=(-0.02, 0.12), xytext=(-0.02, 0.88),
              xycoords="axes fraction", textcoords="axes fraction",
              arrowprops=dict(arrowstyle="-|>", color="0.3", lw=0.9,
                              shrinkA=0, shrinkB=0, mutation_scale=8),
              annotation_clip=False)
ax_m.text(-0.06, 0.5, "better", transform=ax_m.transAxes, rotation=90,
          ha="center", va="center", color="0.3")

# ── Panel letters ────────────────────────────────────────────────────────────

axS[0].text(-0.17, 1.0, "(a)", transform=axS[0].transAxes, va="top", ha="left")
ax_m.text(-0.17, 1.0, "(b)", transform=ax_m.transAxes, va="top", ha="left")

fig.savefig("fig3_orientation.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig.savefig("fig3_orientation.png", dpi=600, bbox_inches="tight", transparent=True)
plt.show()

# ── SI forest figure: 2x2 thickness / roughness / orientation / density ──────

def make_si_forest_figure(savename="fig_si_forest.png"):
    """
    Export the stacked SI forest: (a) orientation, (b) density.

    Each panel shows the free per-energy box-whisker spread behind the
    joint-fit points of both models. The x-limits are exactly the fit
    bounds of the per-energy inversion (GAMMA_FIT_BOUNDS, RHO_FIT_BOUNDS),
    so the displayed window is the allowed parameter space and any pile-up
    of the free clouds against an edge reads as bound clipping.
    """
    fig, (ax_g, ax_r) = plt.subplots(
        2, 1, figsize=(3.35, 3.2),
        gridspec_kw=dict(hspace=0.45),
        sharey=True,
    )

    def _boxes(ax, clouds, free_d, dft_d, xlabel, xbounds):
        for s in slab_order:
            _free_box(ax, clouds[s], ypos[s])
            y = ypos[s]
            fv, fe = free_d[s]
            dv, de = dft_d[s]
            # ax.errorbar(fv, y + 0.18, xerr=fe, marker="s", color=C_FREE,
            #             ecolor=C_FREE, capsize=1.5, ms=3, ls="none", zorder=6)
            ax.errorbar(dv, y - 0.18, xerr=de*4, marker="s", color=C_DFT,
                        ecolor=C_DFT, capsize=1.5, ms=3, ls="none", zorder=6)
        ax.set_xlabel(xlabel, labelpad=1)
        ax.set_xlim(*xbounds)

    _boxes(ax_g, clouds_g, g_free_pt,
           {s: (g_dft[s], g_dft_e[s]) for s in slab_order},
           r"$\gamma$ (deg)", np.degrees(GAMMA_FIT_BOUNDS))
    ax_g.axvline(MAGIC_ANGLE_DEG, color="0.35", lw=0.5, zorder=0)

    _boxes(ax_r, clouds_r, r_free_pt,
           {s: (r_dft[s], r_dft_e[s]) for s in slab_order},
           r"$\rho$ (g cm$^{-3}$)", RHO_FIT_BOUNDS)
    ax_r.axvline(RHO_NOMINAL, color="0.35", lw=0.5, zorder=0)

    for ax, letter in ((ax_g, "(a)"), (ax_r, "(b)")):
        ax.set_yticks(list(ypos.values()))
        ax.set_ylim(-0.8, 2.8)
        ax.invert_yaxis()
        ax.set_yticklabels(slab_names, rotation=45)
        ax.text(0.02, 1.02, letter, transform=ax.transAxes,
                va="bottom", ha="left")

    fig.legend(handles=leg_handles, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, -0.08))
    fig.savefig(savename, dpi=600, bbox_inches="tight", transparent=True)
    fig.savefig(savename.replace(".png", ".pdf"), dpi=600,
                bbox_inches="tight", transparent=True)
    plt.show()


make_si_forest_figure()

In [ ]:
"""
Figure 3 (single PRL column): per-energy free tensor fits vs the DFT slab model.

Argument
  The free model fits the optical tensor independently at each probe energy,
  so its raw outputs are the per-energy birefringence and dichroism of each
  slab, each with a fit uncertainty. The DFT slab model reproduces those
  spectra from two parameters per slab (rotation, density) and threads the
  free points with a propagated uncertainty band. The apparent per-energy
  (gamma, density) estimates scatter widely (SI box-and-whisker) while
  the DFT value is far tighter, because a one-stage joint fit to the
  reflectivity beats the two-stage compress-then-fit.

Layout (single column, 3.35 in)
  (a) three stacked spectra (surface / bulk / interface): DFT birefringence
      (solid) and dichroism (dashed) traces + free per-energy points
      (circles / triangles), with a per-slab DFT gamma readout
  (b) MSC bars

make_si_forest_figure() exports the SI figure:
  (a) delta spectra   (b) beta spectra    per-slab, free points vs DFT traces
  (c) orientation     (d) density         forests
  The spectra sit above the forests so the derivation is visible: rho scales
  the tensor magnitude that (a) and (b) display, while gamma sets the split
  between the xx and zz components shown in the main figure.
  Forest x-limits are the fit bounds of the per-energy inversion.

Styling note
  Font sizes, tick sizes, and line defaults come from the global rcParams
  (set_plotting_defaults). Only the SI slab legend overrides sizes.

Prerequisites (fig3 pipeline)
  free_n        per-(slab, energy) tensor fit, with birefringence/_err,
                dichroism/_err, delta/_err, beta/_err, and xx, zz, ixx, izz
  dft_n         per-slab DFT tensor traces (birefringence, dichroism,
                delta, beta) + rotation, rotation_err, density (ufloat)
  slab_results  free joint-fit gamma, density ufloats per slab
  ooc           oriented optical constants (n_xx, n_zz, n_ixx, n_izz vs energy)
  dft_constrained, exp_constrained   GlobalObjectives, for the MSC bars
"""

from matplotlib import gridspec, ticker
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.markers import CARETDOWNBASE, CARETUPBASE
from matplotlib.ticker import FuncFormatter
from scipy.optimize import least_squares
from uncertainties import ufloat

# Okabe-Ito blue and vermillion. The previous blue/green pair sat too close
# in both hue and luminance to separate at print scale; blue against
# vermillion is the widest separation available and stays distinct under
# deuteranopia and protanopia.
C_DFT = "#0072B2"    # blue
C_FREE = "#D55E00"   # vermillion
C_FREE_FILL = to_rgba(C_FREE, 0.20)

LB, UB = 281.0, 290.0
MAGIC_ANGLE_DEG = np.degrees(np.arcsin(np.sqrt(2 / 3)))
RHO_NOMINAL = 1.61
slab_order = ["surface", "bulk", "interface"]
slab_names = ["surf.", "bulk", "int."]

# Slab encodings for the SI spectra panels, where all three slabs share one
# axis and the model colors are already spent on DFT vs free.
SLAB_MK = {"surface": "o", "bulk": "s", "interface": "^"}
SLAB_LS = {"surface": (0, (1, 1.2)), "bulk": "-", "interface": (0, (3, 1.3))}

# Panel y-bounds are computed per slab from both quantities (traces and free
# points including error bars), symmetric about zero, padded by Y_PAD, and
# clipped at Y_CAP so a single huge pre-edge error bar cannot flatten the
# traces. Bars beyond Y_CAP clip visibly, which is honest.
Y_PAD = 1.10
Y_CAP = 4.5e-3

# Fit bounds for the per-energy (gamma, density) inversion. The SI forest
# axes use exactly these limits, so the displayed window IS the allowed
# parameter space and clipping against it is visible.
GAMMA_FIT_BOUNDS = (0.0, np.pi / 2)          # rad
RHO_FIT_BOUNDS = (1.0, 3.0)                  # g cm^-3

# All three spectra panels share one decade; factor it into the common
# y-label rather than printing a per-panel offset.
BIREF_SCALE = 1e-3

# Pathological per-energy free fits worth calling out, as (energy, column).
# Marked graphically only; the caption carries the explanation, and the
# values are printed to console for the text.
#   bulk 283.7 eV     dichroism collapses to zero
#   surface 287.0 eV  dichroism runs off scale
OUTLIER_FLAGS = {
    "bulk": [(283.7, "dichroism")],
    "surface": [(287.0, "dichroism")],
}
FLAG_C = "0.15"

FIG_W, FIG_H = 3.35, 5.8

# ── Per-energy inversion: free tensor -> apparent (gamma, density) ───────────

def _apparent_gamma_density(row, ooc_df, gamma0, density0):
    """
    Fit (gamma, density) to a single energy's free tensor components.

    Mirrors fit_orientation_from_free but for one energy: four observations
    (ixx, izz, xx, zz) against the oriented optical constants at that energy.
    Returns (gamma_rad, density) or None if the inversion fails (near the
    isotropic point birefringence vanishes and the problem is ill-posed,
    which is itself the source of the per-energy spread).
    """
    e = float(row["energy"])
    ixx = float(np.interp(e, ooc_df["energy"], ooc_df["n_ixx"]))
    izz = float(np.interp(e, ooc_df["energy"], ooc_df["n_izz"]))
    xx0 = float(np.interp(e, ooc_df["energy"], ooc_df["n_xx"]))
    zz0 = float(np.interp(e, ooc_df["energy"], ooc_df["n_zz"]))

    obs = np.array([row["ixx"], row["izz"], row["xx"], row["zz"]], dtype=float)
    err = np.array([
        max(abs(row.get("ixx_err", 0.0)), 1e-12),
        max(abs(row.get("izz_err", 0.0)), 1e-12),
        max(abs(row.get("xx_err", 0.0)), 1e-12),
        max(abs(row.get("zz_err", 0.0)), 1e-12),
    ])

    def resid(p):
        g, d = p
        cg, sg = np.cos(g), np.sin(g)
        m_ixx = d * (ixx * (cg + 1) + izz * sg) / 2
        m_xx = d * (xx0 * (cg + 1) + zz0 * sg) / 2
        m_izz = d * (ixx * sg + izz * cg)
        m_zz = d * (xx0 * sg + zz0 * cg)
        return (obs - np.array([m_ixx, m_izz, m_xx, m_zz])) / err

    try:
        res = least_squares(
            resid, x0=[gamma0, density0], method="trf",
            bounds=([GAMMA_FIT_BOUNDS[0], RHO_FIT_BOUNDS[0]],
                    [GAMMA_FIT_BOUNDS[1], RHO_FIT_BOUNDS[1]]),
            max_nfev=200,
        )
    except Exception:
        return None
    if not res.success:
        return None
    return float(res.x[0]), float(res.x[1])


def per_energy_cloud(slab, ooc_df, gamma0, density0):
    """Apparent gamma (deg) and density across this slab's probe energies."""
    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    gammas, dens = [], []
    for _, row in f.iterrows():
        out = _apparent_gamma_density(row, ooc_df, gamma0, density0)
        if out is None:
            continue
        g, d = out
        gammas.append(np.degrees(g))
        dens.append(d)
    return np.array(gammas), np.array(dens)

# ── Forest scalars ───────────────────────────────────────────────────────────

def _pair(value):
    n = value.n if hasattr(value, "n") else float(value)
    s = value.s if hasattr(value, "s") else 0.0
    return float(n), float(s)


def _dft_scalar(slab, col):
    return _pair(dft_n[dft_n["slab"] == slab].iloc[0][col])


g_dft = {s: np.degrees(_dft_scalar(s, "rotation")[0]) for s in slab_order}
g_dft_e = {s: np.degrees(_dft_scalar(s, "rotation")[1]) for s in slab_order}
r_dft = {s: _dft_scalar(s, "density")[0] for s in slab_order}
r_dft_e = {s: _dft_scalar(s, "density")[1] for s in slab_order}

g_free_pt = {s: _pair(slab_results[s]["gamma"]) for s in slab_order}
g_free_pt = {s: (np.degrees(v), np.degrees(e)) for s, (v, e) in g_free_pt.items()}
r_free_pt = {s: _pair(slab_results[s]["density"]) for s in slab_order}

clouds_g, clouds_r = {}, {}
for s in slab_order:
    g0 = slab_results[s]["gamma"].nominal_value
    d0 = slab_results[s]["density"].nominal_value
    clouds_g[s], clouds_r[s] = per_energy_cloud(s, ooc, g0, d0)
    print(
        f"{s}: per-energy gamma n={clouds_g[s].size}, "
        f"IQR={np.subtract(*np.percentile(clouds_g[s], [75, 25])):.2f} deg, "
        f"DFT err={g_dft_e[s]:.2f} deg"
    )

# ── Statistics (same machinery as fig3) ──────────────────────────────────────

def model_selection_stats(objective, *, use_logl=True):
    resid = np.asarray(objective.residuals(), dtype=float)
    n = int(resid.size)
    k = len(objective.varying_parameters())
    chi2 = float(np.sum(resid**2))
    nu = max(n - k, 1)
    if use_logl:
        logl = float(objective.logl())
        aic = 2.0 * k - 2.0 * logl
        bic = k * np.log(n) - 2.0 * logl
    else:
        aic = chi2 + 2.0 * k
        bic = chi2 + k * np.log(n)
    return {"chi2": chi2 / nu, "aic": aic, "bic": bic, "N": k, "npoints": n}


stats = {
    "DFT slab": model_selection_stats(dft_constrained),
    "Free tensor": model_selection_stats(exp_constrained),
}
# stats["DFT slab"]["N"] = 17
# stats["Free tensor"]["N"] = 266
N_CORR = {"DFT slab": 21 * 4, "Free tensor": 21 * 4}

# ── Per-slab spectral y-bounds ───────────────────────────────────────────────

def _panel_bounds(slab, pad=Y_PAD, cap=Y_CAP):
    """Symmetric y-bounds covering biref and dichroism, traces and points."""
    d = dft_n[dft_n["slab"] == slab]
    d = d[(d["energy"] >= LB) & (d["energy"] <= UB)]
    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    vals = [
        d["birefringence"].to_numpy(dtype=float),
        d["dichroism"].to_numpy(dtype=float),
        (f["birefringence"].abs() + f["birefringence_err"].abs()).to_numpy(dtype=float),
        (f["dichroism"].abs() + f["dichroism_err"].abs()).to_numpy(dtype=float),
    ]
    if "birefringence_err" in d.columns:
        vals.append(
            (d["birefringence"].abs() + d["birefringence_err"].abs()).to_numpy(dtype=float)
        )
    if "dichroism_err" in d.columns:
        vals.append(
            (d["dichroism"].abs() + d["dichroism_err"].abs()).to_numpy(dtype=float)
        )
    m = pad * max(float(np.max(np.abs(v))) for v in vals if v.size)
    m = min(m, cap)
    return (-m, m)


spec_bounds = {s: _panel_bounds(s) for s in slab_order}

# ── Shared bits ──────────────────────────────────────────────────────────────

emk = dict(mfc="white", mec=C_FREE, mew=1.0, ls="none", ecolor=C_FREE,
           elinewidth=0.8, capsize=1.5, ms=3.5)

# Ticks display value / BIREF_SCALE; the decade lives in the shared label.
_biref_fmt = FuncFormatter(lambda v, _: f"{v / BIREF_SCALE:.0f}")

DASH_DIC = (0, (2.2, 1.4))

leg_handles = [
    Line2D([], [], color=C_DFT, marker="s", label="DFT Slab"),
    Line2D([], [], color=C_FREE, marker="s", mfc=C_FREE_FILL,
           ls="none", label="Free Slab"),
]

ypos = {s: i for i, s in enumerate(slab_order)}


def _flag_outlier(ax, slab, energy, column):
    """
    Mark a pathological per-energy free fit, graphically only.

    Two idioms, chosen automatically and carrying no text, so the panels
    stay readable and the caption does the explaining. A point inside the
    window gets an open ring. A point past the window gets a filled caret
    seated inside the axes with a stub running to it, which reads as "this
    one continues off scale" without the panel rescaling around it.

    Values are printed to console so the magnitudes are available for the
    manuscript text.
    """
    f = free_n[free_n["slab"] == slab]
    if f.empty:
        return
    row = f.iloc[(f["energy"] - energy).abs().argmin()]
    val = float(row[column])
    err_col = f"{column}_err"
    err = abs(float(row[err_col])) if err_col in f.columns else 0.0
    e = float(row["energy"])

    lo, hi = ax.get_ylim()
    span = hi - lo
    top = val + err > hi
    bot = val - err < lo

    print(f"flagged {slab} {column} at {e:.2f} eV: "
          f"{val / BIREF_SCALE:.2f} +/- {err / BIREF_SCALE:.2f} "
          f"(x{BIREF_SCALE:g}), window +/-{hi / BIREF_SCALE:.1f}"
          f"{'  [off scale]' if (top or bot) else ''}")

    if not (top or bot):
        ax.plot(e, val, "o", ms=9, mfc="none", mec=FLAG_C, mew=0.9, zorder=7)
        return

    # Off-scale: caret seated inside the axes (CARETUPBASE draws upward from
    # its anchor, so anchoring inside keeps it clear of the neighbouring
    # panel and the legend), with a stub leading the eye out of range.
    mk = CARETUPBASE if top else CARETDOWNBASE
    anchor = hi - 0.14 * span if top else lo + 0.14 * span
    stub_from = hi - 0.42 * span if top else lo + 0.42 * span
    ax.plot(e, anchor, marker=mk, ms=7, color=FLAG_C, mfc=FLAG_C,
            ls="none", zorder=8)
    ax.plot([e, e], [stub_from, anchor], color=C_FREE, lw=0.8, zorder=6)


def _free_box(ax, data, position, width=0.55):
    ax.boxplot(
        [data], positions=[position], vert=False, widths=width, whis=0.6827,
        showfliers=False, patch_artist=True, manage_ticks=False,
        medianprops=dict(color=C_FREE),
        whiskerprops=dict(color=C_FREE),
        capprops=dict(color=C_FREE),
        boxprops=dict(facecolor=C_FREE_FILL, edgecolor=C_FREE),
    )

# ── Figure ───────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(FIG_W, FIG_H))
outer = gridspec.GridSpec(
    2, 1, figure=fig, height_ratios=[4.3, 1.0], hspace=0.26,
)
gs_spec = outer[0].subgridspec(3, 1, hspace=0.08)

axS = [fig.add_subplot(gs_spec[i]) for i in range(3)]
for a in axS[:-1]:
    a.sharex(axS[-1])
ax_m = fig.add_subplot(outer[1])


def _spectral(ax, slab, i, last=False):
    d = dft_n[dft_n["slab"] == slab]
    d = d[(d["energy"] >= LB) & (d["energy"] <= UB)].sort_values("energy")
    if "birefringence_err" in d.columns:
        ax.fill_between(d["energy"],
                        d["birefringence"] - d["birefringence_err"],
                        d["birefringence"] + d["birefringence_err"],
                        color=C_DFT, alpha=0.25, lw=0, zorder=2)
    ax.plot(d["energy"], d["birefringence"], "-", color=C_DFT, zorder=3)
    if "dichroism_err" in d.columns:
        ax.fill_between(d["energy"],
                        d["dichroism"] - d["dichroism_err"],
                        d["dichroism"] + d["dichroism_err"],
                        color=C_DFT, alpha=0.18, lw=0, zorder=2)
    ax.plot(d["energy"], d["dichroism"], ls=DASH_DIC, color=C_DFT, zorder=3)

    f = free_n[free_n["slab"] == slab]
    f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
    ax.errorbar(f["energy"], f["birefringence"], yerr=f["birefringence_err"],
                marker="o", zorder=4, **emk)
    ax.errorbar(f["energy"], f["dichroism"], yerr=f["dichroism_err"],
                marker="^", zorder=4, **emk)

    ax.axhline(0, color="0.6", lw=0.5, zorder=1)
    ax.set_xlim(LB, UB)
    ax.set_xticks([282, 284, 286, 288])

    ax.set_ylim(spec_bounds[slab])
    # Tick step adapts to the panel window: 1e-3 up to ~3e-3 half-range,
    # 2e-3 beyond, so wide panels stay labeled without crowding.
    step = 1e-3 if spec_bounds[slab][1] <= 3.2e-3 else 2e-3
    ax.yaxis.set_major_locator(ticker.MultipleLocator(step))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(step / 2))
    ax.yaxis.set_major_formatter(_biref_fmt)   # no per-panel offset text
    ax.text(0.03, 0.93, slab, transform=ax.transAxes, ha="left", va="top",
            color="0.30")
    # DFT orientation readout for this slab.
    gamma = ufloat(g_dft[slab], g_dft_e[slab] * 4)
    ax.text(
        0.97, 0.93,
        rf"$\gamma = {gamma:.1uS}^\circ$",
        transform=ax.transAxes, ha="right", va="top", color=C_DFT,
    )
    # Outlier marks last, so they sit above the traces and the ylim is set.
    for e_flag, col_flag in OUTLIER_FLAGS.get(slab, []):
        _flag_outlier(ax, slab, e_flag, col_flag)

    if not last:
        ax.tick_params(labelbottom=False)


for i, s in enumerate(slab_order):
    _spectral(axS[i], s, i, last=(i == 2))
axS[2].set_xlabel(r"Energy (eV)", labelpad=-1)

# Single shared y-label across the three spectra, scale folded in. Centered on
# the middle panel, which sits at the vertical center of the equal-height stack.
axS[1].set_ylabel(r"$\Delta\delta$, $\Delta\beta$ ($\times 10^{-3}$)")
axS[1].yaxis.set_label_coords(-0.1, 0.5)

# Quantity legend above the stack; model colors are keyed under the MSC panel.
q_handles = [
    Line2D([], [], color="0.2", ls="-", marker="o", mfc="white", ms=3.5,
           label=r"$\Delta\delta$ (biref.)"),
    Line2D([], [], color="0.2", ls=DASH_DIC, marker="^", mfc="white", ms=3.5,
           label=r"$\Delta\beta$ (dichro.)"),
]
axS[0].legend(handles=q_handles, loc="lower center", bbox_to_anchor=(0.5, 1.0),
              ncol=2, frameon=False, handlelength=1.8,
              columnspacing=1.0, handletextpad=0.5, borderpad=0.1)

# ── MSC bars (carried from fig3) ─────────────────────────────────────────────

METRICS = [r"$\chi^2_\nu$", "N", "AIC", "BIC"]
BAR_W, GROUP_GAP, INTRA_GAP = 0.30, 0.85, 0.04
models = list(stats)
bar_colors = [C_DFT, C_FREE]
mkeys = ["chi2", "N", "aic", "bic"]
gc = np.arange(4) * GROUP_GAP
offs = np.linspace(-(BAR_W + INTRA_GAP) / 2, (BAR_W + INTRA_GAP) / 2, 2)
norms = {mk: max(stats[m][mk] for m in models) for mk in mkeys}
for gi, mk in enumerate(mkeys):
    for bi, (m, col) in enumerate(zip(models, bar_colors)):
        raw = stats[m][mk]
        norm = raw / norms[mk]
        x = gc[gi] + offs[bi]
        if mk == "N":
            corr = N_CORR[m] / norms[mk]
            ax_m.bar(x, corr, BAR_W, color=col, alpha=0.9, lw=0,
                     hatch="/////", edgecolor="white")
            ax_m.bar(x, norm - corr, BAR_W, bottom=corr, color=col, alpha=0.9, lw=0)
            raw -= N_CORR[m]
        else:
            ax_m.bar(x, norm, BAR_W, color=col, alpha=0.9, lw=0)
        ax_m.text(x, norm + 0.02, f"{raw:.1f}" if mk == "chi2" else f"{raw:.0f}",
                  ha="center", va="bottom")
ax_m.set_ylim(0, 1.5)
ax_m.set_xticks(gc)
ax_m.set_xticklabels(METRICS)
ax_m.set_yticklabels([])
ax_m.axhline(1.0, color="0.65", lw=0.5, ls="--")
# add space between MSC label and the plot for the arrow
ax_m.set_ylabel("MSC", labelpad=17)

# Two-entry model legend below the MSC panel.
ax_m.legend(handles=leg_handles, loc="upper center", bbox_to_anchor=(0.5, -0.30),
            ncol=2, frameon=False)

# "lower is better" arrow in the left margin.
ax_m.annotate("", xy=(-0.02, 0.12), xytext=(-0.02, 0.88),
              xycoords="axes fraction", textcoords="axes fraction",
              arrowprops=dict(arrowstyle="-|>", color="0.3", lw=0.9,
                              shrinkA=0, shrinkB=0, mutation_scale=8),
              annotation_clip=False)
ax_m.text(-0.06, 0.5, "better", transform=ax_m.transAxes, rotation=90,
          ha="center", va="center", color="0.3")

# ── Panel letters ────────────────────────────────────────────────────────────

axS[0].text(-0.17, 1.0, "(a)", transform=axS[0].transAxes, va="top", ha="left")
ax_m.text(-0.17, 1.0, "(b)", transform=ax_m.transAxes, va="top", ha="left")

fig.savefig("fig3_orientation.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig.savefig("fig3_orientation.png", dpi=600, bbox_inches="tight", transparent=True)
plt.show()

# ── SI figure: delta / beta spectra above the gamma / density forests ────────

def make_si_forest_figure(savename="fig_si_forest.png"):
    """
    Export the SI figure: (a) delta, (b) beta, (c) orientation, (d) density.

    The spectra sit above the forests so the derivation chain is visible.
    Density scales the whole tensor, so it is read off the magnitude of the
    delta and beta traces in (a) and (b); orientation is read off the split
    between components (the main figure's birefringence and dichroism).
    All three slabs share each spectra axis, distinguished by marker and
    line style, since the model colors already encode DFT vs free.

    Forest x-limits are exactly the fit bounds of the per-energy inversion
    (GAMMA_FIT_BOUNDS, RHO_FIT_BOUNDS), so the displayed window is the
    allowed parameter space and any pile-up of the free clouds against an
    edge reads as bound clipping.
    """
    fig = plt.figure(figsize=(3.35, 5.0))
    outer = gridspec.GridSpec(
        2, 1, figure=fig, height_ratios=[1.55, 1.0], hspace=0.42,
    )
    gs_spec = outer[0].subgridspec(2, 1, hspace=0.10)
    gs_for = outer[1].subgridspec(2, 1, hspace=0.45)

    ax_d = fig.add_subplot(gs_spec[0])
    ax_b = fig.add_subplot(gs_spec[1])
    ax_b.sharex(ax_d)
    ax_g = fig.add_subplot(gs_for[0])
    ax_r = fig.add_subplot(gs_for[1])

    def _optical(ax, col, ylabel, last):
        err_col = f"{col}_err"
        for s in slab_order:
            d = dft_n[dft_n["slab"] == s]
            d = d[(d["energy"] >= LB) & (d["energy"] <= UB)].sort_values("energy")
            ax.plot(d["energy"], d[col], ls=SLAB_LS[s], color=C_DFT,
                    lw=1.0, zorder=3)
            f = free_n[free_n["slab"] == s]
            f = f[(f["energy"] >= LB) & (f["energy"] <= UB)]
            ax.errorbar(
                f["energy"], f[col],
                yerr=f[err_col] if err_col in f.columns else None,
                marker=SLAB_MK[s], ms=3, mfc="white", mec=C_FREE, mew=0.9,
                ls="none", ecolor=C_FREE, elinewidth=0.7, capsize=1.2, zorder=4,
            )
        ax.set_xlim(LB, UB)
        ax.set_xticks([282, 284, 286, 288])
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_formatter(_biref_fmt)
        if not last:
            ax.tick_params(labelbottom=False)

    _optical(ax_d, "delta", r"$\delta$ ($\times 10^{-3}$)", last=False)
    _optical(ax_b, "beta", r"$\beta$ ($\times 10^{-3}$)", last=True)
    ax_b.set_xlabel(r"Energy (eV)", labelpad=-1)

    # Slab key for the spectra panels only.
    slab_h = [
        Line2D([], [], color="0.25", ls=SLAB_LS[s], marker=SLAB_MK[s],
               mfc="white", ms=3, label=n)
        for s, n in zip(slab_order, slab_names)
    ]
    ax_d.legend(handles=slab_h, loc="lower center", bbox_to_anchor=(0.5, 1.0),
                ncol=3, frameon=False, handlelength=1.8, columnspacing=1.0,
                handletextpad=0.4, borderpad=0.1)

    def _boxes(ax, clouds, free_d, dft_d, xlabel, xbounds):
        for s in slab_order:
            _free_box(ax, clouds[s], ypos[s])
            y = ypos[s]
            fv, fe = free_d[s]
            dv, de = dft_d[s]
            # ax.errorbar(fv, y + 0.18, xerr=fe, marker="s", color=C_FREE,
            #             ecolor=C_FREE, capsize=1.5, ms=3, ls="none", zorder=6)
            ax.errorbar(dv, y - 0.18, xerr=de * 4, marker="s", color=C_DFT,
                        ecolor=C_DFT, capsize=1.5, ms=3, ls="none", zorder=6)
        ax.set_xlabel(xlabel, labelpad=1)
        ax.set_xlim(*xbounds)

    _boxes(ax_g, clouds_g, g_free_pt,
           {s: (g_dft[s], g_dft_e[s]) for s in slab_order},
           r"$\gamma$ (deg)", np.degrees(GAMMA_FIT_BOUNDS))
    ax_g.axvline(MAGIC_ANGLE_DEG, color="0.35", lw=0.5, zorder=0)

    _boxes(ax_r, clouds_r, r_free_pt,
           {s: (r_dft[s], r_dft_e[s]) for s in slab_order},
           r"$\rho$ (g cm$^{-3}$)", RHO_FIT_BOUNDS)
    ax_r.axvline(RHO_NOMINAL, color="0.35", lw=0.5, zorder=0)

    for ax in (ax_g, ax_r):
        ax.set_yticks(list(ypos.values()))
        ax.set_ylim(-0.8, 2.8)
        ax.invert_yaxis()
        # No rotation: at this row height rotated labels collide.
        ax.set_yticklabels(slab_names)

    for ax, letter in ((ax_d, "(a)"), (ax_b, "(b)"),
                       (ax_g, "(c)"), (ax_r, "(d)")):
        ax.text(-0.17, 1.02, letter, transform=ax.transAxes,
                va="bottom", ha="left")

    fig.legend(handles=leg_handles, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, -0.05))
    fig.savefig(savename, dpi=600, bbox_inches="tight", transparent=True)
    fig.savefig(savename.replace(".png", ".pdf"), dpi=600,
                bbox_inches="tight", transparent=True)
    plt.show()


make_si_forest_figure()

## Refloxide comparison (new)

Adds the refloxide multi-energy UniTensorSLD fit
(`refloxide/examples/real_data_repl.py`) as a structural comparison table only --
NOT a fourth trace in the delta/beta dispersion-curve panels above. Those panels
compare per-energy independently-fit tensor components (`dft_n`/`free_n`), but the
refloxide fit uses ONE shared structure across all 21 energies (only the
tabulated optical-constant lookup varies with energy, not density/rotation), so
there is no per-energy tensor value to plot alongside them -- a thickness/
roughness/density/rotation table against the SAME anchor energy (283.7 eV) is the
honest comparison here.

In [ ]:
from utils import models_root
from utils.refloxide_view import load_refloxide_fit

REFLOXIDE_FIT_PATH = models_root / "xrr/znpc/refloxide/refit-refloxide.pkl"
if not REFLOXIDE_FIT_PATH.exists():
    raise FileNotFoundError(
        f"missing {REFLOXIDE_FIT_PATH} -- run refloxide/examples/real_data_repl.py "
        "(through the CurveFitter.fit cell) first to produce this pickle"
    )

refloxide_objective = load_refloxide_fit(REFLOXIDE_FIT_PATH)
refloxide_structure = refloxide_objective.model.structure
print(
    f"loaded refloxide fit: {len(refloxide_objective.varying_parameters())} "
    "varying parameters"
)


In [ ]:
# role -> (refloxide slab name, legacy DFT/Free slab name)
ROLE_MAP = {
    "surface": ("ZnPc_surface", "Surface"),
    "bulk": ("ZnPc_bulk", "ZnPc"),
    "interface": ("ZnPc_interface", "Contamination"),
}

ANCHOR_ENERGY = 283.7


def _objective_at(bundle, energy):
    """Objective in a pyref GlobalObjective whose model.energy is closest to `energy`."""
    energies = [float(o.model.energy) for o in bundle.objectives]
    idx = min(range(len(energies)), key=lambda i: abs(energies[i] - energy))
    return bundle.objectives[idx]


def _slab_by_name(structure, name):
    return next(s for s in structure if s.name.split("_")[0] == name or s.name == name)


dft_anchor = _objective_at(dft_constrained, ANCHOR_ENERGY)
free_anchor = _objective_at(exp_constrained, ANCHOR_ENERGY)

rows = []
for role, (rx_name, legacy_name) in ROLE_MAP.items():
    rx_slab = refloxide_structure.slab(rx_name)
    dft_slab = _slab_by_name(dft_anchor.model.structure, legacy_name)
    free_slab = _slab_by_name(free_anchor.model.structure, legacy_name)
    for param in ("thick", "rough"):
        rows.append({
            "role": role, "param": param,
            "refloxide": float(getattr(rx_slab, param).value),
            "dft": float(getattr(dft_slab, param).value),
            "free": float(getattr(free_slab, param).value),
        })
    for param in ("density", "rotation"):
        rows.append({
            "role": role, "param": param,
            "refloxide": float(getattr(rx_slab.sld, param).value),
            "dft": float(getattr(dft_slab.sld, param).value),
            "free": float(getattr(free_slab.sld, param).value)
            if hasattr(free_slab.sld, param) else float("nan"),
        })

comparison_table = pd.DataFrame(rows)
display(comparison_table)
